# IMDB 링크 예측(누출 없음): FISF, 균일 결측 — 분할 1/3

**왜**: 11(ACM·DBLP 링크 예측)에서 FISF·HetGFD + 주입을 돌렸고, IMDB만 비어 있음 → 표를 채움. HetGFD는 08_3(같은 프로토콜) 결과를 씀.

**설정**: 링크 예측, 원 코드 설정, **학습 그래프에서만 보간(누출 없음)**, 결측률 0.995, IMDB 균일 결측. 08_3·06처럼 1단계 상위 5개 조합을 seed 0–2에서 HGT 전체 격자로 재평가해서 고름(`--refine-top 5`).
방법: `fisf` (원 FISF, 관계 구분 없음; α 5 × γ 4, β_s 0.9)

- **1단계 (선택)**: seed 0–2, HGT 1개 설정(2층, 학습률 0.01)으로 하이퍼파라미터가 있는 방법의 후보를 모두 돌려 **검증 점수 평균**으로 선택
  (HetGFD α·β 40개, PCFI+VF α·γ 10개, kNN k 4개, SVD 계수 2개 — 이 노트북에서 돌리는 방법만)
- **2단계 (최종)**: seed 0–9, HGT 격자(층 1–3 × 학습률 4개)
- 진행 순서: 1단계 전체 → 2단계 seed 0–4 → 2단계 seed 5–9 (중간에 끊겨도 쓸 만한 결과부터 나오게)
- **GPU 병렬**: GPU마다 작업자 1개가 공용 대기열에서 (설정, 단계, seed) 단위 일을 가져감. 한 설정의 1단계 seed 3개가 끝나면 그 설정의 2단계 시작

**분할**: 12_1 FISF 균일, 12_2 HetGFD + 주입 구조적, 12_3 HetGFD + 주입 균일 (FISF 구조적은 로컬) — 폴더가 달라 동시에 실행 가능.

**끊겨도 이어서 돌리기**: 결과는 학습 1회마다 `results/*.jsonl`에 바로 저장되고, 다시 띄우면 끝난 것은 건너뜁니다.
**새 작업(job)을 잡을 때마다 셀 1 → 셀 8을 위에서부터 차례로 실행**하면 됩니다 (셀 7 점검은 처음 한 번만; 전부 여러 번 실행해도 안전). 셀 번호는 01 노트북과 같게 맞춤 (셀 4는 없음: HGNN-AC 코드가 필요 없음).
셀 8은 먼저 이전에 돌던 프로세스(예전 노트북으로 띄운 것, 멈춘 작업자 등)를 자동으로 멈추고, GPU별 작업자가 이미 살아 있으면(최근 2분 안에 신호) 그대로 두고, 없는 GPU에만 새로 띄웁니다.
**결과 전달**: 셀 12가 `WORK/export/`에 CSV(요약·선택·전체 실행)를 만들고 요약 CSV를 출력 → 그 출력을 복사해서 Claude에게 붙여넣기.


In [ ]:
# ===== 셀 1: 설정 =====
import os, pathlib, sys, json, time
WORK = os.path.expanduser("~/hetgfd_hgt_imdb_lp_1")   # 영구 저장 공간 아래 경로 (다른 노트북과 겹치지 않게 따로)
# 01 노트북에서 이미 받아 둔 데이터가 있으면 다시 받지 않고 그대로 씀 (있는 경로를 순서대로 찾음)
SHARED_DATA = [os.path.expanduser("~/hetgfd_hgnnac_sweep/data/preprocessed")]
GPUS = "auto"            # "auto" = 보이는 GPU 전부 (예: 4장). 일부만 쓰려면 [0, 1] 처럼
SEL_SEEDS = [0, 1, 2]
FINAL_SEEDS = list(range(10))
for d in ["src/hetgfd_lib", "logs", "results", "state", "data"]:
    pathlib.Path(WORK, d).mkdir(parents=True, exist_ok=True)
RUN = f"{sys.executable} {WORK}/src/run_hgt_final.py"
ARGS = f"--tasks lp --datasets IMDB --modes entry --methods fisf --impute-train --refine-top 5 --sel-seeds {' '.join(map(str, SEL_SEEDS))} --final-seeds {' '.join(map(str, FINAL_SEEDS))}"
print("WORK =", WORK)

In [ ]:
# ===== 셀 2: 패키지 설치 (01 노트북과 같은 방식; torch는 서버에 있는 것 사용) =====
import sys, subprocess
print(sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric", "gdown", "networkx", "scikit-learn", "scipy", "pandas", "requests"], check=True)
import torch, torch_geometric
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| pyg", torch_geometric.__version__)

In [ ]:
# ===== 셀 3: 데이터셋 (HGNN-AC / MAGNN 전처리 데이터) =====
# 순서: 이미 있음 → SHARED_DATA에 있으면 연결(symlink) → WORK/data/hgnnac.zip(직접 올린 것) → gdown → drive.usercontent 직접 요청
import zipfile, shutil, requests
FILE_ID = "1PqUjvSViICa8yOszqDrw-j96hXVJ0MHR"
data = pathlib.Path(WORK, "data"); pre = data / "preprocessed"; z = data / "hgnnac.zip"
need = ["ACM_processed", "DBLP_processed", "IMDB_processed"]
ready = lambda p: all((p / n / "adjM.npz").exists() for n in need)
ok_zip = lambda p: p.exists() and zipfile.is_zipfile(p)
if not ready(pre):
    for s in map(pathlib.Path, SHARED_DATA):
        if ready(s):
            if pre.is_symlink() or pre.exists(): pre.unlink() if pre.is_symlink() else shutil.rmtree(pre)
            pre.symlink_to(s); print("기존 데이터 연결:", s); break
if not ready(pre):
    def try_gdown():
        import gdown; gdown.download(id=FILE_ID, output=str(z), quiet=False)
    def try_usercontent():
        with requests.get(f"https://drive.usercontent.google.com/download?id={FILE_ID}&export=download&confirm=t", stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(z, "wb") as f:
                for chunk in r.iter_content(1 << 20): f.write(chunk)
    if not ok_zip(z):
        for name, fn in [("gdown", try_gdown), ("usercontent", try_usercontent)]:
            for attempt in range(3):
                try:
                    print(f"[{name}] 시도 {attempt + 1}/3"); fn()
                    if ok_zip(z): break
                    print("  zip이 아님"); z.unlink(missing_ok=True)
                except Exception as e:
                    print(f"  실패: {type(e).__name__}: {str(e)[:200]}"); z.unlink(missing_ok=True)
                time.sleep(5)
            if ok_zip(z): break
    if not ok_zip(z):
        raise SystemExit(f"Google Drive 접속 불가. 브라우저로 https://drive.google.com/file/d/{FILE_ID}/view 를 받아 "
                         f"{data}/hgnnac.zip 으로 올린 뒤 이 셀을 다시 실행")
    tmp = data / "_unzip"
    with zipfile.ZipFile(z) as f: f.extractall(tmp)
    pre.mkdir(exist_ok=True)
    for n in need:
        src = [p for p in tmp.rglob(n) if p.is_dir()][0]
        if (pre / n).exists(): shutil.rmtree(pre / n)
        shutil.move(str(src), pre / n)
    shutil.rmtree(tmp)
for n in need: print(n, sorted(os.listdir(pre / n))[:8])

### 셀 5: 소스 파일 작성 (아래 `%%writefile` 셀 모두 실행 — 새 작업마다 다시 실행해도 됨)
- `hetgfd_lib/`: HetGFD·PCFI+VF·FP, 마스크, HGT 노드 분류(`downstream.py`), 원 코드 링크 예측(`linkpred_orig.py`) — 로컬 재현 코드와 동일
- `gpu_queue.py`: GPU 여러 장이 일을 나눠 갖는 대기열 (파일 기반)
- `run_hgt_final.py`: 실험 본체

In [ ]:
%%writefile {WORK}/src/hetgfd_lib/__init__.py
"""HetGFD reproduction library (copied from src/hetdiff)."""


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/hetgraph.py
"""Full heterogeneous graph in the HetGFD setting (HGNN-AC / MAGNN preprocessing).

Nodes are indexed globally (types stacked in ``node_types`` order). Only one node
type is *attributed* (raw features); all others get virtual features during
diffusion. Relations are raw edge types, stored as symmetric N×N 0/1 matrices.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
from scipy.sparse.csgraph import connected_components

# name, node type names (index order of node_types.npy), attributed type, label type
HGNNAC_SPECS = {
    "ACM": {"types": ["paper", "author", "subject"], "attr": "paper", "label": "paper"},
    "DBLP": {"types": ["author", "paper", "term", "venue"], "attr": "paper", "label": "author"},
    "IMDB": {"types": ["movie", "director", "actor"], "attr": "movie", "label": "movie"},
}


@dataclass
class HetGraph:
    name: str
    node_types: list[str]
    counts: list[int]
    edges: dict[str, sp.csr_matrix]  # edge type name -> symmetric (N, N) 0/1 adjacency
    edge_ends: dict[str, tuple[str, str]]
    attr_type: str
    x: torch.Tensor  # (N_attr, F) raw features of the attributed type
    label_type: str
    y: torch.Tensor
    split: dict[str, torch.Tensor]  # indices local to label_type
    meta: dict = field(default_factory=dict)

    @property
    def num_nodes(self) -> int:
        return sum(self.counts)

    def offset(self, t: str) -> int:
        i = self.node_types.index(t)
        return sum(self.counts[:i])

    def type_slice(self, t: str) -> slice:
        o = self.offset(t)
        return slice(o, o + self.counts[self.node_types.index(t)])

    def type_of_nodes(self) -> np.ndarray:
        return np.repeat(np.arange(len(self.node_types)), self.counts)

    def block(self, et: str) -> sp.csr_matrix:
        """Bipartite block (src-type rows, dst-type cols) of an edge type."""
        s, d = self.edge_ends[et]
        return self.edges[et][self.type_slice(s)][:, self.type_slice(d)].tocsr()


def _load_feat(p: Path) -> np.ndarray:
    if p.with_suffix(".npz").exists():
        return sp.load_npz(p.with_suffix(".npz")).toarray()
    return np.load(p.with_suffix(".npy"))


def load_hgnnac(
    name: str, root: str = "data/raw/HGNNAC/preprocessed", lcc: bool = True, same_type_directed: bool = False
) -> HetGraph:
    """``same_type_directed``: keep same-type relations (ACM paper–paper citations) as the raw directed
    adjacency incl. self-loops, as the original HetGFD code does; otherwise symmetrise them."""
    spec = HGNNAC_SPECS[name]
    d = Path(root) / f"{name}_processed"
    nt = np.load(d / "node_types.npy")
    raw = sp.load_npz(d / "adjM.npz").tocsr().astype(np.float32)
    adj = ((raw + raw.T) > 0).astype(np.float32).tocsr()
    adj.setdiag(0)
    adj.eliminate_zeros()
    types = spec["types"]
    ai, li = types.index(spec["attr"]), types.index(spec["label"])
    x = _load_feat(d / f"features_{ai}").astype(np.float32)
    y = np.load(d / "labels.npy")
    sp_idx = np.load(d / "train_val_test_idx.npz")
    split = {k.replace("_idx", ""): sp_idx[k] for k in sp_idx.files}

    keep = np.ones(adj.shape[0], dtype=bool)
    if lcc:
        # LCC over edges between *different* node types. This reproduces HetGFD Table 4 exactly
        # (ACM: 4014 / 7157 / 56 nodes; a plain LCC including P-P citations keeps 4017 / 7165 / 58).
        coo = adj.tocoo()
        cross = nt[coo.row] != nt[coo.col]
        adj_x = sp.csr_matrix((coo.data[cross], (coo.row[cross], coo.col[cross])), shape=adj.shape)
        _, comp = connected_components(adj_x, directed=False)
        keep = comp == np.bincount(comp).argmax()
    new_id = -np.ones(adj.shape[0], dtype=np.int64)
    new_id[keep] = np.arange(keep.sum())
    adj = adj[keep][:, keep].tocsr()
    nt_k = nt[keep]
    counts = [int((nt_k == i).sum()) for i in range(len(types))]

    # local re-indexing of attributed / labelled types
    def local_keep(ti):
        return keep[nt == ti]

    x = x[local_keep(ai)]
    # raw features shipped for other types (not used by HetGFD's imputation; optional downstream input)
    other_x = {}
    for ti, t in enumerate(types):
        if ti != ai and (d / f"features_{ti}.npz").exists() or (ti != ai and (d / f"features_{ti}.npy").exists()):
            other_x[t] = torch.from_numpy(_load_feat(d / f"features_{ti}").astype(np.float32)[local_keep(ti)])
    lk = local_keep(li)
    y = y[lk]
    local_map = -np.ones(lk.size, dtype=np.int64)
    local_map[lk] = np.arange(lk.sum())
    split = {k: local_map[v][local_map[v] >= 0] for k, v in split.items()}

    # split adjacency into edge types by endpoint node types
    edges, ends = {}, {}
    coo = sp.triu(adj).tocoo()
    ts, td = nt_k[coo.row], nt_k[coo.col]
    for a in range(len(types)):
        for b in range(a, len(types)):
            sel = ((ts == a) & (td == b)) | ((ts == b) & (td == a))
            if not sel.any():
                continue
            m = sp.csr_matrix((np.ones(sel.sum(), np.float32), (coo.row[sel], coo.col[sel])), shape=adj.shape)
            m = ((m + m.T) > 0).astype(np.float32).tocsr()
            et = f"{types[a][0].upper()}-{types[b][0].upper()}"
            edges[et], ends[et] = m, (types[a], types[b])
    directed = set()
    if same_type_directed:
        rk = raw[keep][:, keep].tocoo()
        for et, (s_t, d_t) in ends.items():
            if s_t != d_t:
                continue
            ti = types.index(s_t)
            sel = (nt_k[rk.row] == ti) & (nt_k[rk.col] == ti)
            edges[et] = sp.csr_matrix((np.ones(sel.sum(), np.float32), (rk.row[sel], rk.col[sel])), shape=adj.shape)
            directed.add(et)

    return HetGraph(
        name=name,
        node_types=types,
        counts=counts,
        edges=edges,
        edge_ends=ends,
        attr_type=spec["attr"],
        x=torch.from_numpy(x),
        label_type=spec["label"],
        y=torch.from_numpy(y).long(),
        split={k: torch.from_numpy(v) for k, v in split.items()},
        meta={"lcc_removed": int((~keep).sum()), "other_x": other_x, "directed": directed},
    )


def metapath_blocks(g: HetGraph) -> dict[str, list[sp.csr_matrix]]:
    """Length-2 meta-paths attr→B→attr (and direct attr–attr edges) as chains of bipartite blocks."""
    out = {}
    a = g.attr_type
    for et, (s, d) in g.edge_ends.items():
        if s == a and d == a:
            out[et] = [g.block(et)]
            continue
        if a not in (s, d):
            continue
        other = d if s == a else s
        sl_a, sl_o = g.type_slice(a), g.type_slice(other)
        ab = g.edges[et][sl_a][:, sl_o].tocsr()
        name = f"{a[0].upper()}-{other[0].upper()}-{a[0].upper()}"
        out[name] = [ab, ab.T.tocsr()]
    return out


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/hetgfd.py
"""Re-implementation of HetGFD (Um et al., ICLR 2025) from the paper (official code is not released).

Pipeline (Sec. 4):
  1. virtual (zero) features for non-attributed nodes
  2. preliminary diffusion with Ā = Σ_r |E_r|^{-1} A_r (row-stochastic, known rows fixed)
  3. edge-type-wise homophily H(r) on the pre-imputed matrix → ranking r*_1..r*_R
  4. pseudo-confidence ξ = α^S, S = SPD to nearest source on W = Σ_r β^{-(k_r-1)} A_r
  5. relation-aware diffusion with W̄_ij = β^{k_r-1} ξ_j / ξ_i, row-normalised, known rows fixed

After row normalisation ξ_i cancels, so T_ij ∝ β^{k-1} α^{S_j - S_i}; we evaluate it as a
grouped softmax in log space (|S_j - S_i| ≤ w_ij keeps it bounded, but α^S itself underflows).

Special cases: α = β = 1 → FP+VF;  β = 1 → PCFI+VF without PCFI's inter-channel step.
"""

from __future__ import annotations

import math

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from scipy.sparse.csgraph import dijkstra
from torch_geometric.utils import softmax as grouped_softmax

from hetgfd_lib.hetgraph import HetGraph


def _global_init(g: HetGraph, observed: torch.Tensor, device):
    n, f = g.num_nodes, g.x.shape[1]
    sl = g.type_slice(g.attr_type)
    x0 = torch.zeros(n, f, device=device)
    known = torch.zeros(n, f, dtype=torch.bool, device=device)
    x0[sl] = torch.where(observed, g.x.to(device), torch.zeros_like(x0[sl]))
    known[sl] = observed
    return x0, known, sl


def _csr(a: sp.spmatrix, device) -> torch.Tensor:
    a = sp.csr_matrix(a, dtype=np.float32)
    return torch.sparse_csr_tensor(
        torch.from_numpy(a.indptr).long(), torch.from_numpy(a.indices).long(), torch.from_numpy(a.data), size=a.shape
    ).to(device)


def _row_normalize(a: sp.spmatrix) -> sp.csr_matrix:
    deg = np.asarray(a.sum(1)).ravel()
    return sp.csr_matrix(sp.diags(np.where(deg > 0, 1.0 / np.maximum(deg, 1e-30), 0.0)) @ a)


def _fixed_point(P: torch.Tensor, x0: torch.Tensor, known: torch.Tensor, K: int) -> torch.Tensor:
    x = x0.clone()
    for _ in range(K):
        x = torch.where(known, x0, P @ x)
    return x


def preliminary_diffusion(g: HetGraph, observed: torch.Tensor, K: int = 100, device=None) -> torch.Tensor:
    """Returns the pre-imputed matrix X̄ for *all* nodes (N, F)."""
    directed = g.meta.get("directed", set())
    # |E_r| = number of edges per direction (a symmetric matrix stores each edge twice)
    abar = sum(a / (a.nnz if et in directed else a.nnz / 2) for et, a in g.edges.items())
    x0, known, _ = _global_init(g, observed, device)
    return _fixed_point(_csr(_row_normalize(abar), device), x0, known, K)


def _mean_pair_cos(xn: torch.Tensor, i: torch.Tensor, j: torch.Tensor, chunk: int = 16384) -> float:
    tot = 0.0
    for s in range(0, len(i), chunk):
        tot += float((xn[i[s : s + chunk]] * xn[j[s : s + chunk]]).sum(1).sum())
    return tot / max(len(i), 1)


def edge_type_homophily(x: torch.Tensor, g: HetGraph, num_samples: int = 200_000, seed: int = 0) -> dict[str, float]:
    """Definition 1: mean cos over r-edges / mean cos over random node pairs in V (chunked: O(chunk·F) memory)."""
    rng = np.random.default_rng(seed)
    xn = F.normalize(x, dim=1)
    n = x.shape[0]
    a = torch.from_numpy(rng.integers(n, size=num_samples)).to(x.device)
    b = torch.from_numpy(rng.integers(n, size=num_samples)).to(x.device)
    keep = a != b
    rand = _mean_pair_cos(xn, a[keep], b[keep])
    out = {}
    directed = g.meta.get("directed", set())
    for et, adj in g.edges.items():
        coo = (adj if et in directed else sp.triu(adj)).tocoo()
        idx = rng.choice(coo.nnz, size=min(num_samples, coo.nnz), replace=False)
        i = torch.from_numpy(coo.row[idx]).long().to(x.device)
        j = torch.from_numpy(coo.col[idx]).long().to(x.device)
        out[et] = _mean_pair_cos(xn, i, j) / rand if abs(rand) > 1e-12 else float("nan")
    return out


def rank_edge_types(h: dict[str, float]) -> dict[str, int]:
    order = sorted(h, key=lambda k: -np.nan_to_num(h[k], nan=-np.inf))
    return {et: k + 1 for k, et in enumerate(order)}  # k = 1 is the most homophilic


def _source_distances(w_pc: sp.csr_matrix, sources: np.ndarray) -> np.ndarray:
    if sources.size == 0:
        return np.full(w_pc.shape[0], np.inf)
    # directed: W[i, j] is the edge i -> j (row i aggregates from j), as in the original networkx DiGraph;
    # identical to the undirected search when every relation is stored symmetrically
    return dijkstra(w_pc, directed=True, indices=sources, min_only=True)


def relation_aware_diffusion(
    g: HetGraph,
    observed: torch.Tensor,
    rank: dict[str, int],
    alpha: float,
    beta: float,
    K: int = 100,
    device=None,
    channel_chunk: int = 256,
    return_dist: bool = False,
    edge_count_norm: bool = False,
):
    """With ``return_dist`` also returns S (source distances) for all nodes: (N,) structural, (N, F) uniform.
    ``edge_count_norm``: also multiply the final transition weight of every r-type edge by 1/|E_r| (edges per
    direction), i.e. the commented-out line in the original hfp.compute_edge_weight_from_pc."""
    x0, known, sl = _global_init(g, observed, device)
    directed = g.meta.get("directed", set())
    n = g.num_nodes
    w_pc = sp.csr_matrix((n, n), dtype=np.float64)
    rows, cols, logw = [], [], []
    for et, adj in g.edges.items():
        k = rank[et]
        w_pc = w_pc + adj.astype(np.float64) * beta ** (-(k - 1))
        coo = adj.tocoo()
        rows.append(coo.row)
        cols.append(coo.col)
        lw = (k - 1) * math.log(beta)
        if edge_count_norm:
            lw -= math.log(adj.nnz if et in directed else adj.nnz / 2)
        logw.append(np.full(coo.nnz, lw))
    dst = torch.from_numpy(np.concatenate(rows)).long().to(device)  # i (receiver)
    src = torch.from_numpy(np.concatenate(cols)).long().to(device)  # j (sender)
    logw = torch.from_numpy(np.concatenate(logw)).float().to(device)
    off = sl.start
    log_alpha = math.log(alpha)

    def dist_for(source_rows: np.ndarray) -> torch.Tensor:
        s = _source_distances(w_pc, source_rows + off)
        return torch.from_numpy(np.nan_to_num(s, posinf=1e12)).float().to(device)

    obs_np = observed.cpu().numpy()
    structural = bool((obs_np == obs_np[:, :1]).all())
    if structural:
        s = dist_for(np.flatnonzero(obs_np[:, 0]))
        logits = logw + (s[src] - s[dst]) * log_alpha
        vals = grouped_softmax(logits, dst, num_nodes=n)
        P = torch.sparse_coo_tensor(torch.stack([dst, src]), vals, (n, n)).coalesce().to_sparse_csr()
        full = _fixed_point(P, x0, known, K)
        return (full, s) if return_dist else full[sl]

    # uniform missing: one SPD / transition per channel, processed in chunks
    out = torch.empty(n, x0.shape[1], device=device)
    s_all = torch.empty(n, x0.shape[1], device=device) if return_dist else None
    cache: dict[bytes, torch.Tensor] = {}
    for c0 in range(0, x0.shape[1], channel_chunk):
        cs = range(c0, min(c0 + channel_chunk, x0.shape[1]))
        dists = []
        for c in cs:
            key = np.packbits(obs_np[:, c]).tobytes()
            if key not in cache:
                cache[key] = dist_for(np.flatnonzero(obs_np[:, c]))
            dists.append(cache[key])
        s = torch.stack(dists, 1)  # (n, C)
        logits = logw[:, None] + (s[src] - s[dst]) * log_alpha
        vals = grouped_softmax(logits, dst, num_nodes=n)  # (E, C)
        xc0, kc = x0[:, c0 : cs.stop], known[:, c0 : cs.stop]
        x = xc0.clone()
        for _ in range(K):
            msg = torch.zeros_like(x).index_add_(0, dst, vals * x[src])
            x = torch.where(kc, xc0, msg)
        out[:, c0 : cs.stop] = x
        if return_dist:
            s_all[:, c0 : cs.stop] = s
        cache.clear()
    return (out, s_all) if return_dist else out[sl]


def hetgfd(
    g: HetGraph,
    observed: torch.Tensor,
    alpha: float = 0.5,
    beta: float = 0.5,
    K: int = 100,
    device=None,
    seed: int = 0,
    rank: dict[str, int] | None = None,
    edge_count_norm: bool = False,
) -> tuple[torch.Tensor, dict]:
    """Full HetGFD. Pass ``rank`` to override the homophily ranking (e.g. random / utility ranking)."""
    info = {}
    if rank is None:
        xbar = preliminary_diffusion(g, observed, K, device)
        h = edge_type_homophily(xbar, g, seed=seed)
        rank = rank_edge_types(h)
        info["H"] = h
    info["rank"] = rank
    info["edge_count_norm"] = edge_count_norm
    return relation_aware_diffusion(g, observed, rank, alpha, beta, K, device, edge_count_norm=edge_count_norm), info


def reversed_homophily_rank(g: HetGraph, observed: torch.Tensor, K: int = 100, device=None, seed: int = 0) -> dict[str, int]:
    """Edge-type ranking in the OPPOSITE order of edge-type-wise homophily (least homophilic first)."""
    h = edge_type_homophily(preliminary_diffusion(g, observed, K, device), g, seed=seed)
    r = rank_edge_types(h)
    return {et: len(r) + 1 - k for et, k in r.items()}


def fp_vf(g: HetGraph, observed: torch.Tensor, K: int = 100, device=None) -> torch.Tensor:
    """FP + virtual features: unweighted diffusion on the union graph (α = β = 1)."""
    rank = {et: 1 for et in g.edges}
    return relation_aware_diffusion(g, observed, rank, 1.0, 1.0, K, device)


def pcfi_vf(g: HetGraph, observed: torch.Tensor, alpha: float = 0.5, K: int = 100, device=None) -> torch.Tensor:
    """PCFI + virtual features, channel-wise part only (β = 1)."""
    rank = {et: 1 for et in g.edges}
    return relation_aware_diffusion(g, observed, rank, alpha, 1.0, K, device)


# ---------------------------------------------------------------------------------------------
# Baselines exactly as in the original HetGFD code (third_party/neurips1-master: fp.py, cfp.py)
# ---------------------------------------------------------------------------------------------


def fp_orig(g: HetGraph, observed: torch.Tensor, K: int = 100, device=None) -> torch.Tensor:
    """fp.py: unweighted union graph, symmetric normalisation D^-1/2 A D^-1/2 with D from column sums."""
    x0, known, sl = _global_init(g, observed, device)
    u = sp.csr_matrix(sum(a for a in g.edges.values()), dtype=np.float64)
    u.data[:] = 1.0
    deg = np.asarray(u.sum(0)).ravel()
    dinv = np.where(deg > 0, deg ** -0.5, 0.0)
    P = sp.diags(dinv) @ u @ sp.diags(dinv)
    return _fixed_point(_csr(P, device), x0, known, K)[sl]


def pcfi_orig(g: HetGraph, observed: torch.Tensor, alpha: float = 0.5, gamma: float = 0.02, K: int = 100, device=None):
    """cfp.py: pseudo-confidence diffusion on the unweighted graph, then the inter-channel step
    X += γ (1 − β^S) ⊙ ((β^S ⊙ (X − mean)) · Corr),  with β = α (``self.beta = alpha`` in the original)."""
    rank = {et: 1 for et in g.edges}
    out, s = relation_aware_diffusion(g, observed, rank, alpha, 1.0, K, device, return_dist=True)
    s = s[:, None] if s.dim() == 1 else s
    conf = alpha ** s.clamp_max(1e6)
    cor = torch.corrcoef(out.T).nan_to_num().fill_diagonal_(0)
    a2 = (conf * (out - out.mean(0))) @ cor
    out = out + gamma * (1 - conf) * a2
    return out[g.type_slice(g.attr_type)]


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/masking.py
"""Missing-feature masks.

Convention: a mask is a bool tensor of shape (N, d) where True means *observed*.

- ``node`` mode (structural missing): whole feature rows are missing.
- ``entry`` mode (uniform missing): individual entries are missing.
"""

from __future__ import annotations

import torch


def make_observed_mask(
    n: int, d: int, missing_rate: float, mode: str = "node", seed: int = 0, device=None
) -> torch.Tensor:
    gen = torch.Generator().manual_seed(seed)
    if mode == "node":
        rows = torch.rand(n, generator=gen) >= missing_rate
        mask = rows[:, None].expand(n, d).clone()
    elif mode == "entry":
        mask = torch.rand(n, d, generator=gen) >= missing_rate
    else:
        raise ValueError(f"unknown mask mode: {mode}")
    return mask.to(device)


def split_probe(
    observed: torch.Tensor, probe_frac: float, mode: str = "node", seed: int = 0
) -> tuple[torch.Tensor, torch.Tensor]:
    """Split observed entries into (train, probe). Truly missing entries are never touched."""
    # salt the seed so the probe draw is independent of make_observed_mask(seed=seed)
    gen = torch.Generator().manual_seed(seed + 7_919_000)
    if mode == "node":
        obs_rows = observed.all(dim=1).cpu()
        probe_rows = obs_rows & (torch.rand(observed.shape[0], generator=gen) < probe_frac)
        probe = probe_rows[:, None].expand_as(observed).to(observed.device)
    elif mode == "entry":
        probe = observed & (torch.rand(observed.shape, generator=gen) < probe_frac).to(observed.device)
    else:
        raise ValueError(f"unknown mask mode: {mode}")
    train = observed & ~probe
    return train, probe


def make_observed_mask_orig(n: int, d: int, missing_rate: float, mode: str = "node", seed: int = 0, device=None):
    """utils.get_missing_feature_mask of the original HetGFD code (global torch seed + Bernoulli)."""
    torch.manual_seed(seed)
    if mode == "node":
        m = torch.bernoulli(torch.Tensor([1 - missing_rate]).repeat(n)).bool().unsqueeze(1).repeat(1, d)
    elif mode == "entry":
        m = torch.bernoulli(torch.Tensor([1 - missing_rate]).repeat(n, d)).bool()
    else:
        raise ValueError(f"unknown mask mode: {mode}")
    return m.to(device)


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/downstream.py
"""Downstream evaluation: semi-supervised node classification with HGT (HetGFD App. B.3).

Imputed features go to the attributed type; non-attributed nodes get one-hot features.
HetGFD Sec. 5.1 ("one-hot node features for each node type") is ambiguous; per-node one-hot
(``node_onehot``, default) matches the paper's Zero / Full numbers on ACM better than a
per-type constant (``type_onehot``): Zero 82.3 vs 80.6 (paper 82.75), Full 93.2 vs 92.1 (92.5).
Model selection on validation Macro-F1, early stopping (patience 200, max 1000 epochs).
"""

from __future__ import annotations

import copy
import itertools

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

from hetgfd_lib.hetgraph import HetGraph


def build_heterodata(g: HetGraph, x_attr: torch.Tensor, nonattr_feat: str = "node_onehot") -> HeteroData:
    data = HeteroData()
    T = len(g.node_types)
    for i, t in enumerate(g.node_types):
        n = g.counts[i]
        if t == g.attr_type:
            data[t].x = x_attr.float().cpu()
        elif nonattr_feat == "type_onehot":
            data[t].x = F.one_hot(torch.full((n,), i), T).float()
        elif nonattr_feat == "node_onehot":
            data[t].x = torch.eye(n)
        elif nonattr_feat == "raw":  # dataset-provided features of non-attributed types, if any
            ox = g.meta.get("other_x", {})
            data[t].x = ox[t] if t in ox else F.one_hot(torch.full((n,), i), T).float()
        elif nonattr_feat.startswith("raw:"):  # raw features only for the listed types
            ox, use = g.meta.get("other_x", {}), nonattr_feat[4:].split(",")
            data[t].x = ox[t] if t in use and t in ox else torch.eye(n)
        else:
            raise ValueError(nonattr_feat)
    for et, (s, d) in g.edge_ends.items():
        blk = g.block(et).tocoo()
        ei = torch.tensor(np.vstack([blk.row, blk.col]), dtype=torch.long)
        data[s, f"{et}", d].edge_index = ei
        if s != d:
            data[d, f"{et}_rev", s].edge_index = ei.flip(0)
    return data


class HGT(nn.Module):
    def __init__(
        self, data: HeteroData, hidden: int, out: int, layers: int, heads: int, target: str, dropout: float, act: str = "gelu"
    ):
        super().__init__()
        self.act = {"gelu": F.gelu, "relu": F.relu}[act]
        self.lin = nn.ModuleDict({t: nn.Linear(data[t].x.shape[1], hidden) for t in data.node_types})
        self.convs = nn.ModuleList([HGTConv(hidden, hidden, data.metadata(), heads) for _ in range(layers)])
        self.out = nn.Linear(hidden, out)
        self.target, self.dropout = target, dropout

    def forward(self, x_dict, ei_dict):
        h = {t: self.act(self.lin[t](x)) for t, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, ei_dict)
            h = {t: F.dropout(v, self.dropout, self.training) for t, v in h.items()}
        return self.out(h[self.target])


def _f1(logits, y):
    p = logits.argmax(1).cpu().numpy()
    y = y.cpu().numpy()
    return f1_score(y, p, average="macro"), f1_score(y, p, average="micro")


def _val_score(logits, y, select):
    if select == "acc":  # the original code selects the epoch by validation accuracy
        return float((logits.argmax(1) == y).float().mean())
    return _f1(logits, y)[0]


def train_eval(
    data: HeteroData,
    g: HetGraph,
    layers: int,
    lr: float,
    hidden: int = 64,
    heads: int = 2,
    dropout: float = 0.2,
    epochs: int = 1000,
    patience: int = 200,
    weight_decay: float = 0.0,
    seed: int = 0,
    device=None,
    act: str = "gelu",
    select: str = "macro",
) -> dict:
    torch.manual_seed(seed)
    data = data.to(device)
    y = g.y.to(device)
    tr, va, te = (g.split[k].to(device) for k in ("train", "val", "test"))
    model = HGT(data, hidden, int(y.max()) + 1, layers, heads, g.label_type, dropout, act).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best, best_state, bad = -1.0, None, 0
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(data.x_dict, data.edge_index_dict)
        F.cross_entropy(out[tr], y[tr]).backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)
        v = _val_score(out[va], y[va], select)
        if v > best:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)
    ma, mi = _f1(out[te], y[te])
    del model, opt, out, best_state
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # hand cached blocks back so parallel runs on the same GPU do not starve
    return {"val_macro": best, "test_macro": ma, "test_micro": mi, "layers": layers, "lr": lr}  # val_macro = selection score


def evaluate_features(
    g: HetGraph,
    x_attr: torch.Tensor,
    grid_layers=(1, 2, 3),
    grid_lr=(0.1, 0.01, 0.001, 0.0001),
    seed: int = 0,
    device=None,
    nonattr_feat: str = "node_onehot",
    **kw,
) -> dict:
    """Grid search on validation Macro-F1, report test scores of the selected config."""
    data = build_heterodata(g, x_attr, nonattr_feat)
    runs = [train_eval(data, g, L, lr, seed=seed, device=device, **kw) for L, lr in itertools.product(grid_layers, grid_lr)]
    return max(runs, key=lambda r: r["val_macro"])


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/linkpred_orig.py
"""Link prediction exactly as in the original HetGFD code (third_party/neurips1-master:
run_link_final.py, train_test_split_edges_custom.py, hgt_train_eval.py, models.HGTEncoder*),
plus a corrected evaluation.

Original protocol
  - split of the target edge type: val 5 %, test 10 %, train 85 % (torch.randperm with torch.manual_seed(seed))
  - val/test negatives: uniform non-edges of a (src_type × dst_type) grid sized with the node counts
    BEFORE the largest-connected-component step (IMDB 4278×2081, ACM 4019×7167, DBLP 4057×14328)
  - imputation on the full graph (val/test target edges included); HGT message passing on train edges only
  - encoder: HGT, per-node one-hot for non-attributed types, z = cat([z_src_type, z_dst_type])
  - training: GAE.recon_loss with destination ids offset by n_src; negatives = PyG negative_sampling over
    all nodes of z (any src/dst type pair)
  - evaluation (``orig``): GAE.test(z, pos, neg) WITHOUT the n_src offset → the second index points into
    the src-type block of z (e.g. IMDB scores movie·movie pairs)

Corrected evaluation (``fixed``): destination ids offset by n_src, negatives drawn from the post-LCC grid.
Both evaluations are tracked in the same training run (training does not depend on them); epoch and
(layers, lr) are selected separately per evaluation by its own validation AUC.
"""

from __future__ import annotations

import copy
import itertools
import math
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.nn import HGTConv
from torch_geometric.utils import negative_sampling

from hetgfd_lib.downstream import build_heterodata
from hetgfd_lib.hetgraph import HetGraph
import dataclasses
import scipy.sparse as sp

TARGET_EDGE = {"ACM": "P-A", "DBLP": "A-P", "IMDB": "M-D"}
RAW_GRID = {"ACM": (4019, 7167), "DBLP": (4057, 14328), "IMDB": (4278, 2081)}  # pre-LCC node counts
EPS = 1e-15


@dataclass
class OrigSplit:
    et: str
    src_type: str
    dst_type: str
    n_src: int
    n_dst: int
    train_p: torch.Tensor  # (2, E) local (src, dst)
    eval: dict  # mode -> {"val": (pos, neg), "test": (pos, neg)} as *indices into z*
    g_train: HetGraph


def make_orig_split(g: HetGraph, dataset: str, seed: int) -> OrigSplit:
    et = TARGET_EDGE[dataset]
    s_t, d_t = g.edge_ends[et]
    n_src, n_dst = g.counts[g.node_types.index(s_t)], g.counts[g.node_types.index(d_t)]
    blk = g.block(et).tocoo()
    row, col = torch.from_numpy(blk.row).long(), torch.from_numpy(blk.col).long()

    # --- train_test_split_edges_custom (val_ratio 0.05, test_ratio 0.1) ---
    torch.manual_seed(seed)
    np.random.seed(seed)
    n_v, n_t = int(math.floor(0.05 * row.numel())), int(math.floor(0.1 * row.numel()))
    perm = torch.randperm(row.numel())
    row, col = row[perm], col[perm]
    val_p = torch.stack([row[:n_v], col[:n_v]])
    test_p = torch.stack([row[n_v : n_v + n_t], col[n_v : n_v + n_t]])
    train_p = torch.stack([row[n_v + n_t :], col[n_v + n_t :]])
    grid = torch.ones(*RAW_GRID[dataset], dtype=torch.uint8)
    grid[row, col] = 0
    nr, nc = grid.nonzero(as_tuple=False).t()
    p = torch.randperm(nr.numel())[: n_v + n_t]
    nr, nc = nr[p], nc[p]
    val_n, test_n = torch.stack([nr[:n_v], nc[:n_v]]), torch.stack([nr[n_v:], nc[n_v:]])

    # --- corrected negatives: post-LCC grid, separate generator (does not disturb the original draw) ---
    gen = torch.Generator().manual_seed(10_000 + seed)
    pos_keys = set((row * n_dst + col).tolist())
    fixed_neg, need = [], n_v + n_t
    while len(fixed_neg) < need:
        cand = (torch.randint(n_src, (2 * need,), generator=gen) * n_dst + torch.randint(n_dst, (2 * need,), generator=gen)).tolist()
        for c in cand:
            if c not in pos_keys and c not in fixed_neg:
                fixed_neg.append(c)
                if len(fixed_neg) == need:
                    break
    fk = torch.tensor(fixed_neg)
    fval_n, ftest_n = torch.stack([fk[:n_v] // n_dst, fk[:n_v] % n_dst]), torch.stack([fk[n_v:] // n_dst, fk[n_v:] % n_dst])

    off = torch.tensor([[0], [n_src]])
    ev = {
        "orig": {"val": (val_p, val_n), "test": (test_p, test_n)},  # no offset, as in run_link_final.py
        "fixed": {"val": (val_p + off, fval_n + off), "test": (test_p + off, ftest_n + off)},
    }

    # message-passing graph: target edges restricted to the training positives (both directions)
    n = g.num_nodes
    o_s, o_d = g.offset(s_t), g.offset(d_t)
    a = sp.csr_matrix((np.ones(train_p.shape[1], np.float32), (train_p[0].numpy() + o_s, train_p[1].numpy() + o_d)), shape=(n, n))
    a = ((a + a.T) > 0).astype(np.float32).tocsr()
    g_train = dataclasses.replace(g, edges={**g.edges, et: a})
    return OrigSplit(et, s_t, d_t, n_src, n_dst, train_p, ev, g_train)


class HGTEncoderOrig(nn.Module):
    """models.HGTEncoder*: Linear → ReLU per type, HGTConv stack, z = cat([z_src, z_dst])."""

    def __init__(self, data, hidden: int, layers: int, heads: int, src_type: str, dst_type: str):
        super().__init__()
        self.lin = nn.ModuleDict({t: nn.Linear(data[t].x.shape[1], hidden) for t in data.node_types})
        self.convs = nn.ModuleList([HGTConv(hidden, hidden, data.metadata(), heads) for _ in range(layers)])
        self.src_type, self.dst_type = src_type, dst_type

    def forward(self, x_dict, ei_dict):
        h = {t: F.relu(self.lin[t](x)) for t, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, ei_dict)
        return torch.cat([h[self.src_type], h[self.dst_type]], 0)


def _decode(z, e):
    return torch.sigmoid((z[e[0]] * z[e[1]]).sum(-1))


def _auc_ap(z, pos, neg):
    s = torch.cat([_decode(z, pos), _decode(z, neg)]).cpu().numpy()
    y = np.r_[np.ones(pos.shape[1]), np.zeros(neg.shape[1])]
    return roc_auc_score(y, s), average_precision_score(y, s)


def train_eval_orig(data, sp_: OrigSplit, layers, lr, hidden=64, heads=8, wd=1e-4, epochs=1000, patience=200, seed=0, device=None):
    torch.manual_seed(seed)
    data = data.to(device)
    model = HGTEncoderOrig(data, hidden, layers, heads, sp_.src_type, sp_.dst_type).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    pos = torch.stack([sp_.train_p[0], sp_.train_p[1] + sp_.n_src]).to(device)
    n_z = sp_.n_src + sp_.n_dst
    ev = {m: {k: (p.to(device), n.to(device)) for k, (p, n) in d.items()} for m, d in sp_.eval.items()}
    hist = {m: [] for m in ev}
    best = {m: None for m in ev}
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        z = model(data.x_dict, data.edge_index_dict)
        neg = negative_sampling(pos, num_nodes=n_z)  # GAE.recon_loss default
        loss = -torch.log(_decode(z, pos) + EPS).mean() - torch.log(1 - _decode(z, neg) + EPS).mean()
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, data.edge_index_dict)
            for m, d in ev.items():
                v = _auc_ap(z, *d["val"])[0]
                if not hist[m] or v > max(hist[m]):
                    t_auc, t_ap = _auc_ap(z, *d["test"])
                    best[m] = {"val_auc": v, "test_auc": t_auc, "test_ap": t_ap, "epoch": epoch}
                hist[m].append(v)
        # original early-stopping rule; stop only once it holds for both evaluations
        if epoch > patience and all(max(h[-patience:]) <= max(h[:-patience]) for h in hist.values()):
            break
    del model, opt, z
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # hand cached blocks back so parallel runs on the same GPU do not starve
    return {m: {**best[m], "layers": layers, "lr": lr} for m in ev}


def evaluate_features_orig(sp_: OrigSplit, x_attr, grid_layers=(1, 2, 3), grid_lr=(0.1, 0.01, 0.001, 0.0001), seed=0, device=None, **kw):
    """Grid search; the best (layers, lr) is chosen separately for each evaluation mode by its val AUC."""
    data = build_heterodata(sp_.g_train, x_attr, "node_onehot")
    runs = [train_eval_orig(data, sp_, L, lr, seed=seed, device=device, **kw) for L, lr in itertools.product(grid_layers, grid_lr)]
    return {m: max((r[m] for r in runs), key=lambda r: r["val_auc"]) for m in ("orig", "fixed")}


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/baselines.py
"""Structure-agnostic imputation baselines used in HetGFD (App. B.1)."""

from __future__ import annotations

import torch
import torch.nn.functional as F


def knn_impute(x: torch.Tensor, observed: torch.Tensor, k: int, chunk: int = 2048) -> torch.Tensor:
    """HetGFD's custom kNN: k edges per node by cosine similarity (on zero-filled features),
    missing entries filled with the mean of the neighbours' *observed* values in that channel."""
    x0 = torch.where(observed, x, torch.zeros_like(x))
    xn = F.normalize(x0, dim=1)
    m = observed.float()
    out = x0.clone()
    n = x.shape[0]
    for s in range(0, n, chunk):
        sim = xn[s : s + chunk] @ xn.T
        idx = torch.arange(s, min(s + chunk, n), device=x.device)
        sim[torch.arange(len(idx)), idx] = -float("inf")  # no self
        nb = sim.topk(k, dim=1).indices  # (c, k)
        num = (x0[nb] * m[nb]).sum(1)
        den = m[nb].sum(1)
        fill = torch.where(den > 0, num / den.clamp_min(1.0), torch.zeros_like(num))
        out[s : s + chunk] = torch.where(observed[s : s + chunk], x0[s : s + chunk], fill)
    return out


def iterative_svd_impute(
    x: torch.Tensor, observed: torch.Tensor, rank: int, max_iters: int = 100, tol: float = 1e-5
) -> torch.Tensor:
    """fancyimpute.IterativeSVD re-implemented in torch (zero init, rank-r SVD reconstruction,
    observed entries clamped, stop when relative change < tol)."""
    x_obs = torch.where(observed, x, torch.zeros_like(x))
    cur = x_obs.clone()
    rank = max(1, min(rank, min(x.shape) - 1))
    for _ in range(max_iters):
        if rank >= min(x.shape) // 2:
            u, s, vh = torch.linalg.svd(cur, full_matrices=False)
            rec = (u[:, :rank] * s[:rank]) @ vh[:rank]
        else:
            u, s, v = torch.svd_lowrank(cur, q=rank, niter=4)
            rec = (u * s) @ v.T
        new = torch.where(observed, x_obs, rec)
        delta = (new - cur).norm() / cur.norm().clamp_min(1e-12)
        cur = new
        if delta < tol:
            break
    return cur


In [ ]:
%%writefile {WORK}/src/hetgfd_lib/inject.py
"""HetGFD + synthetic feature injection (the idea of FISF, "Propagate and Inject", ICML 2025, ported to heterogeneous graphs).

1. relation-aware diffusion (HetGFD, given edge-type rank, alpha, beta)
2. the int(F * gamma) channels with the lowest variance over the attributed nodes are selected (at missing rate 0.995
   most channels have no observed non-zero value at all and stay constant)
3. for every selected channel one synthetic value ~ U(0, 1) is placed at one random node and treated as known
     where = "attr": a node of the attributed type (as FISF, where every node carries features)
     where = "virt": a node of a non-attributed type (author / subject / term / venue ...): one injection reaches
                     all of its attributed neighbours
4. the selected channels are diffused again with the HetGFD transition plus the synthetic source:
     logit(i <- j) = (k_r - 1) log beta + (S_j - S_i) log alpha + (S'_j - S'_i) log beta_s
   (S = distance to the real observed entries, S' = distance to the synthetic node of that channel; FISF:
   pseudo-confidence alpha^S * beta_s^S'); the other channels keep the HetGFD output.
Original FISF = this with all edge types equal (rank 1, beta 1) and where = "attr".
"""

from __future__ import annotations

import math

import numpy as np
import scipy.sparse as sp
import torch
from torch_geometric.utils import softmax as grouped_softmax

from hetgfd_lib.hetgfd import _global_init, _source_distances, relation_aware_diffusion
from hetgfd_lib.hetgraph import HetGraph


def hetgfd_inject(g: HetGraph, observed: torch.Tensor, rank: dict, alpha: float, beta: float, gamma: float, beta_s: float,
                  where: str = "attr", K: int = 100, device=None, seed: int = 0, chunk: int = 128) -> torch.Tensor:
    n, f = g.num_nodes, g.x.shape[1]
    full, s_real = relation_aware_diffusion(g, observed, rank, alpha, beta, K, device, return_dist=True)
    x0, known, sl = _global_init(g, observed, device)
    out = full[sl].clone()
    n_low = int(f * gamma)
    if n_low == 0:
        return out
    low = torch.topk(out.var(0), n_low, largest=False).indices.cpu().numpy()

    # synthetic sources (one per selected channel)
    rng = np.random.default_rng(seed)
    if where == "attr":  # an attributed node whose entry in that channel is not observed (never overwrite real values)
        obs_np = observed.cpu().numpy()
        nodes = np.array([sl.start + rng.choice(np.flatnonzero(~obs_np[:, c])) for c in low])
    else:
        cand = np.setdiff1d(np.arange(n), np.arange(sl.start, sl.stop))
        nodes = rng.choice(cand, size=n_low, replace=True)
    vals = torch.from_numpy(rng.random(n_low)).float().to(device)

    # edge list / relation log-weights and the weighted graph for distances, as in relation_aware_diffusion
    directed = g.meta.get("directed", set())
    w_pc = sp.csr_matrix((n, n), dtype=np.float64)
    rows, cols, logw = [], [], []
    for et, adj in g.edges.items():
        k = rank[et]
        w_pc = w_pc + adj.astype(np.float64) * beta ** (-(k - 1))
        coo = adj.tocoo()
        rows.append(coo.row); cols.append(coo.col)
        logw.append(np.full(coo.nnz, (k - 1) * math.log(beta)))
    dst = torch.from_numpy(np.concatenate(rows)).long().to(device)
    src = torch.from_numpy(np.concatenate(cols)).long().to(device)
    logw = torch.from_numpy(np.concatenate(logw)).float().to(device)
    la, lb = math.log(alpha), math.log(beta_s)
    s_real = s_real if s_real.dim() == 2 else s_real[:, None].expand(n, f)

    for c0 in range(0, n_low, chunk):
        ch = low[c0:c0 + chunk]
        nd = nodes[c0:c0 + chunk]
        s_syn = torch.stack([torch.from_numpy(np.nan_to_num(_source_distances(w_pc, np.array([v])), posinf=1e12)).float()
                             for v in nd], 1).to(device)  # (n, C)
        sr = s_real[:, ch]
        logits = logw[:, None] + (sr[src] - sr[dst]) * la + (s_syn[src] - s_syn[dst]) * lb
        vals_e = grouped_softmax(logits, dst, num_nodes=n)  # (E, C)
        xc0 = x0[:, ch].clone()
        kc = known[:, ch].clone()
        cols_ = torch.arange(len(ch), device=device)
        nd_t = torch.from_numpy(nd).long().to(device)
        xc0[nd_t, cols_] = vals[c0:c0 + chunk]
        kc[nd_t, cols_] = True
        x = xc0.clone()
        for _ in range(K):
            msg = torch.zeros_like(x).index_add_(0, dst, vals_e * x[src])
            x = torch.where(kc, xc0, msg)
        out[:, torch.from_numpy(ch).long().to(device)] = x[sl]
    return out


In [ ]:
%%writefile {WORK}/src/gpu_queue.py
"""Tiny file-based work queue so that one worker process per GPU can share a list of work units.

- every launch has a LAUNCH_ID; a worker claims a unit by creating state/claims/<LAUNCH_ID>/<unit> with O_EXCL,
  so two workers of the same launch never run the same unit (claims of older, dead launches are ignored)
- what is *done* is never read from claims, only from the result files, so a job killed at any moment
  loses at most the run in progress
- every worker refreshes state/<worker>.json every 30 s (heartbeat) with its launch id and current run
"""

from __future__ import annotations

import json
import os
import threading
import time
from pathlib import Path

ALIVE_SEC = 120


class Queue:
    def __init__(self, state_dir: Path, launch_id: str, worker: str):
        self.state, self.launch, self.worker = Path(state_dir), launch_id, worker
        self.claims = self.state / "claims" / launch_id
        self.claims.mkdir(parents=True, exist_ok=True)
        self.status = {"current": "starting"}
        self._lock = threading.Lock()  # the heartbeat thread and the main loop both write the state file
        threading.Thread(target=self._beat, daemon=True).start()

    # ---- heartbeat
    def _beat(self):
        while True:
            try:
                self.beat()
            except OSError:  # e.g. a short storage hiccup: try again at the next beat
                pass
            time.sleep(30)

    def beat(self):
        with self._lock:
            atomic_write(self.state / f"{self.worker}.json",
                         json.dumps({"worker": self.worker, "launch": self.launch, "pid": os.getpid(),
                                     "host": os.uname().nodename, "time": time.time(),
                                     "time_str": time.strftime("%Y-%m-%d %H:%M:%S"), **self.status}))

    def worker_alive(self, worker: str) -> bool:
        try:
            s = json.loads((self.state / f"{worker}.json").read_text())
        except Exception:
            return False
        return s.get("launch") == self.launch and s.get("current") != "finished" and time.time() - s["time"] < ALIVE_SEC

    # ---- claims
    def _cpath(self, unit: str) -> Path:
        return self.claims / unit.replace("/", "_")

    def try_claim(self, unit: str) -> bool:
        try:
            fd = os.open(self._cpath(unit), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            return False
        with os.fdopen(fd, "w") as f:
            json.dump({"worker": self.worker, "status": "running", "time": time.time()}, f)
        return True

    def release(self, unit: str):
        """mark the unit as attempted (results decide whether it is done)."""
        p = self._cpath(unit)
        atomic_write(p, json.dumps({"worker": self.worker, "status": "attempted", "time": time.time()}))

    def being_worked_on(self, unit: str) -> bool:
        """claimed in this launch by a live worker that is still running it."""
        try:
            c = json.loads(self._cpath(unit).read_text())
        except Exception:
            return False
        return c.get("status") == "running" and self.worker_alive(c["worker"])


def atomic_write(path: Path, text: str):
    tmp = path.with_name(f".{path.name}.{os.getpid()}.{threading.get_ident()}.tmp")
    tmp.write_text(text)
    os.replace(tmp, path)


def append_jsonl(path: Path, row: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write("\n" + json.dumps(row) + "\n")  # leading newline: never glue onto a half-written line
        f.flush()
        os.fsync(f.fileno())


def read_jsonl(paths) -> list:
    rows = []
    for p in paths:
        p = Path(p)
        if not p.exists():
            continue
        for line in p.read_text().splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:  # empty line or half-written line from a killed job
                pass
    return rows


def run_queue(q: Queue, units, is_done, is_ready, run_unit, deps, log, poll=60):
    """units: ordered unit ids. deps(u) -> unit ids that must finish first (for the 'waiting' decision)."""
    while True:
        progressed, waiting = False, False
        for u in units:
            if is_done(u):
                continue
            if not is_ready(u):
                if any(q.being_worked_on(d) for d in deps(u)):
                    waiting = True  # another GPU is still producing what this unit needs
                continue
            if not q.try_claim(u):
                continue
            q.status["current"] = u
            q.beat()
            try:
                run_unit(u)
            finally:
                q.release(u)
            progressed = True
            break
        if progressed:
            continue
        if waiting:
            q.status["current"] = "waiting for other GPUs (stage 1)"
            time.sleep(poll)
            continue
        break
    q.status["current"] = "finished"
    q.beat()
    log.info(f"==== worker {q.worker} finished (no more work)")


In [ ]:
%%writefile {WORK}/src/run_hgt_final.py
"""HetGFD Table 1 (node classification) / Table 2 (link prediction) with HGT, one unified protocol
(original-code settings, = local exp08 / exp09), alpha·beta chosen on the validation set, many seeds.

Settings: task {nc, lp} x dataset {ACM, DBLP, IMDB} x missing type {node (structural), entry (uniform)}, r_m 0.995.

  stage 1 (selection): seeds SEL_SEEDS, one HGT config (2 layers, lr 0.01)
      HetGFD : paper grid alpha {0.9,...,0.1} x beta {0.99,...,0.05}       (40)
      PCFI+VF: alpha {0.9,0.7,0.5,0.3,0.1} x gamma {0.02, 0.1}             (10)
      -> pick the combination with the best mean validation score
         (nc: validation accuracy = the original code's selection score; lp: corrected-evaluation val AUC)
      kNN    : k {1,3,5,10}; SVD: rank {F/5, F-1}                          (only if in --methods)
  stage 2 (final): seeds FINAL_SEEDS, full HGT grid layers {1,2,3} x lr {0.1,0.01,0.001,0.0001}
      every method in --methods (default: HetGFD, PCFI+VF (selected), FP (original, symmetric), Zero;
      also available: Mean, kNN, SVD)

--hgt orig (default): HGT as in the original code (8 heads, no dropout, ReLU, epoch/config chosen by val accuracy).
--hgt text (node classification only): HGT as written in the paper (2 heads, dropout 0.2, GELU, chosen by val
  macro-F1; non-attributed DBLP nodes get per-type one-hot) = local exp05. Imputers, masks and graph stay as above.
--impute-train (lp only): imputation sees only the training graph (the target edges of the training split), so the
  validation / test edges cannot leak into the imputed features; the original code imputes on the full graph.
--refine-top K (default 0 = off): "stage 3" between selection and final — the K best combinations of stage 1
  (by mean validation score) of every tuned method are re-evaluated on the selection seeds with the FULL HGT grid,
  and the final choice is the best of those (same conditions as the final evaluation). Used where the single
  stage-1 HGT config is not representative (IMDB link prediction).
CSV: results/summary_<setting>.csv (final-stage mean/std per method) is rewritten after every run.

Parallel GPUs: one worker process per GPU (launched with CUDA_VISIBLE_DEVICES=<gpu>) pulls work units from a
shared file-based queue (gpu_queue.py). A unit is (setting, stage, seed): stage-1 units (50 runs each) for every
setting first, then stage-2 units (4 runs each) seeds 0-4, then 5-9; a stage-2 unit becomes ready once all
stage-1 units of its setting are finished (possibly on other GPUs).
Every finished run is appended to results/<task>_<ds>_<mode>/stage<k>_seed<s>.jsonl (one file per unit, so
workers never write the same file); finished runs are skipped on restart, a half-written last line is ignored.
"""

from __future__ import annotations

import argparse
import dataclasses
import itertools
import json
import logging
import os
import sys
import threading
import time
import traceback
from pathlib import Path

import numpy as np
import torch

ap = argparse.ArgumentParser()
ap.add_argument("--workdir", required=True)
ap.add_argument("--worker", required=True, help="name, e.g. gpu0")
ap.add_argument("--launch", required=True, help="launch id shared by the workers of one launch")
ap.add_argument("--tasks", nargs="+", default=["nc", "lp"], choices=["nc", "lp"])
ap.add_argument("--datasets", nargs="+", default=["ACM", "DBLP", "IMDB"])
ap.add_argument("--modes", nargs="+", default=["node", "entry"])
ap.add_argument("--sel-seeds", type=int, nargs="+", default=[0, 1, 2])
ap.add_argument("--final-seeds", type=int, nargs="+", default=list(range(10)))
ap.add_argument("--methods", nargs="+", default=["hetgfd", "pcfi", "fp", "zero"],
                choices=["hetgfd", "pcfi", "fp", "zero", "mean", "knn", "svd", "hetgfd_umse", "hetgfd_ucos"]
                + [f"hetgfd_o{i}" for i in range(1, 7)] + ["fisf", "hetgfd_inj_attr", "hetgfd_inj_virt"])
ap.add_argument("--hgt", default="orig", choices=["orig", "text"])
ap.add_argument("--refine-top", type=int, default=0, help="re-evaluate the top-K stage-1 combos with the full HGT grid")
ap.add_argument("--impute-train", action="store_true",
                help="lp: impute on the training graph only (val/test target edges removed). The original code imputes on "
                     "the full graph, so the held-out edges leak into the imputed features")
ap.add_argument("--smoke", action="store_true", help="tiny run to check the pipeline (separate results)")
args = ap.parse_args()

W = Path(args.workdir)
# early "alive" signal, written before the slow imports (torch_geometric etc.), so the notebook can tell
# a worker that is still starting from one that never started; the queue's heartbeat replaces it
_st = W / ("smoke" if args.smoke else "") / "state"
_st.mkdir(parents=True, exist_ok=True)
(_st / f"{args.worker}.json").write_text(json.dumps({"worker": args.worker, "launch": args.launch, "pid": os.getpid(), "host": os.uname().nodename,
                                                     "time": time.time(), "current": "starting (loading libraries)"}))
sys.path[:0] = [str(W / "src")]
from hetgfd_lib.downstream import evaluate_features  # noqa: E402
from hetgfd_lib.hetgfd import edge_type_homophily, fp_orig, hetgfd, pcfi_orig, preliminary_diffusion, rank_edge_types  # noqa: E402
from hetgfd_lib.hetgraph import load_hgnnac  # noqa: E402
from hetgfd_lib.linkpred_orig import evaluate_features_orig, make_orig_split  # noqa: E402
from hetgfd_lib.masking import make_observed_mask_orig, split_probe  # noqa: E402
from hetgfd_lib.hetgfd import relation_aware_diffusion  # noqa: E402
from hetgfd_lib.inject import hetgfd_inject  # noqa: E402
from hetgfd_lib.baselines import iterative_svd_impute, knn_impute  # noqa: E402
from gpu_queue import Queue, append_jsonl, atomic_write, read_jsonl, run_queue  # noqa: E402

ALPHAS = (0.9, 0.7, 0.5, 0.3, 0.1)
BETAS = (0.99, 0.9, 0.8, 0.5, 0.4, 0.2, 0.1, 0.05)
PCFI_GRID = [(a, gm) for a in (0.9, 0.7, 0.5, 0.3, 0.1) for gm in (0.02, 0.1)]
LAYERS, LRS = (1, 2, 3), (0.1, 0.01, 0.001, 0.0001)
SEL_LAYERS, SEL_LRS = (2,), (0.01,)
FINAL_METHODS = list(args.methods)
KNN_GRID = [(k, 0.0) for k in (1, 3, 5, 10)]
SVD_GRID = [(0.2, 0.0), (1.0, 0.0)]  # p1 = rank / F: 0.2 -> F // 5, 1.0 -> F - 1 (HetGFD paper's SVD grid)
K = 100
RATE = 0.995
HGT_KW = dict(heads=8, dropout=0.0, weight_decay=0.0, act="relu", select="acc", nonattr_feat="node_onehot")  # original code
if args.hgt == "text":  # paper text (local exp05)
    assert args.tasks == ["nc"], "--hgt text is for node classification only"
    HGT_KW = dict(heads=2, dropout=0.2, weight_decay=0.0, act="gelu", select="macro")
NONATTR_TEXT = {"ACM": "node_onehot", "DBLP": "type_onehot", "IMDB": "node_onehot"}
EPOCH_KW = {}
if args.smoke:
    ALPHAS, BETAS, PCFI_GRID = (0.9,), (0.9, 0.1), [(0.5, 0.02)]
    KNN_GRID, SVD_GRID = [(1, 0.0), (5, 0.0)], [(0.2, 0.0)]
    LAYERS, LRS = (1, 2), (0.01,)
    EPOCH_KW = dict(epochs=20, patience=10)

# paper (HetGFD ICLR 2025): Table 5 alpha, beta and Table 1 / 2 numbers (nc: macro, micro F1; lp: AUC, AP)
PAPER_AB = {"nc": {("ACM", "node"): (0.7, 0.2), ("DBLP", "node"): (0.1, 0.4), ("IMDB", "node"): (0.7, 0.1),
                   ("ACM", "entry"): (0.1, 0.4), ("DBLP", "entry"): (0.1, 0.4), ("IMDB", "entry"): (0.3, 0.5)},
            "lp": {("ACM", "node"): (0.1, 0.4), ("DBLP", "node"): (0.5, 0.8), ("IMDB", "node"): (0.9, 0.8),
                   ("ACM", "entry"): (0.1, 0.5), ("DBLP", "entry"): (0.9, 0.9), ("IMDB", "entry"): (0.9, 0.99)}}
PAPER = {
    "nc": {("ACM", "node"): {"mean": (83.25, 83.90), "knn": (82.67, 83.56), "svd": (82.71, 83.20), "zero": (82.75, 83.44), "fp": (83.15, 83.60), "pcfi": (84.47, 85.06), "hetgfd": (85.87, 86.10)},
           ("DBLP", "node"): {"mean": (89.88, 90.74), "knn": (90.37, 91.04), "svd": (90.11, 90.90), "zero": (90.11, 90.90), "fp": (90.30, 91.07), "pcfi": (90.41, 91.14), "hetgfd": (90.88, 91.53)},
           ("IMDB", "node"): {"mean": (45.38, 48.21), "knn": (44.88, 48.22), "svd": (44.28, 47.91), "zero": (44.29, 47.91), "fp": (45.37, 48.30), "pcfi": (46.64, 49.69), "hetgfd": (46.88, 49.81)},
           ("ACM", "entry"): {"mean": (82.76, 83.48), "knn": (81.91, 82.43), "svd": (80.45, 81.52), "zero": (81.85, 82.67), "fp": (83.89, 84.49), "pcfi": (86.26, 86.61), "hetgfd": (88.14, 88.14)},
           ("DBLP", "entry"): {"mean": (90.54, 91.32), "knn": (90.50, 91.20), "svd": (90.33, 91.07), "zero": (90.19, 90.99), "fp": (90.15, 90.91), "pcfi": (90.49, 91.14), "hetgfd": (90.65, 91.40)},
           ("IMDB", "entry"): {"mean": (46.33, 48.32), "knn": (45.81, 48.70), "svd": (44.53, 48.38), "zero": (46.54, 48.49), "fp": (45.58, 48.61), "pcfi": (47.25, 49.94), "hetgfd": (48.57, 50.57)}},
    "lp": {("ACM", "node"): {"mean": (71.64, 71.66), "knn": (72.04, 72.55), "svd": (71.49, 72.29), "zero": (71.65, 71.74), "fp": (73.40, 74.03), "pcfi": (73.41, 73.22), "hetgfd": (78.25, 78.62)},
           ("DBLP", "node"): {"mean": (72.49, 74.20), "knn": (71.96, 69.86), "svd": (72.49, 74.21), "zero": (72.49, 74.21), "fp": (71.58, 70.01), "pcfi": (71.37, 66.78), "hetgfd": (91.94, 91.88)},
           ("IMDB", "node"): {"mean": (91.78, 85.80), "knn": (91.10, 84.44), "svd": (92.48, 86.95), "zero": (92.48, 86.95), "fp": (92.50, 86.99), "pcfi": (91.71, 85.37), "hetgfd": (92.50, 86.99)},
           ("ACM", "entry"): {"mean": (71.98, 72.02), "knn": (71.02, 72.49), "svd": (70.49, 70.70), "zero": (70.69, 70.17), "fp": (73.18, 73.77), "pcfi": (74.94, 73.80), "hetgfd": (76.96, 77.19)},
           ("DBLP", "entry"): {"mean": (72.48, 74.20), "knn": (72.72, 70.29), "svd": (72.48, 74.20), "zero": (72.48, 74.20), "fp": (71.86, 70.03), "pcfi": (70.76, 68.97), "hetgfd": (92.17, 92.12)},
           ("IMDB", "entry"): {"mean": (91.40, 85.33), "knn": (91.15, 84.50), "svd": (92.50, 86.99), "zero": (92.50, 86.99), "fp": (91.52, 85.67), "pcfi": (91.54, 85.70), "hetgfd": (91.95, 86.72)}},
}

RES = W / ("smoke" if args.smoke else "") / "results"
LOGS = W / ("smoke" if args.smoke else "") / "logs"
STATE = W / ("smoke" if args.smoke else "") / "state"
for d in (RES, LOGS, STATE):
    d.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                    handlers=[logging.FileHandler(LOGS / f"{args.worker}.log"), logging.StreamHandler(sys.stdout)])
log = logging.getLogger()
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------- records (one file per unit)
def unit_path(task, ds, mode, stage, seed):
    return RES / f"{task}_{ds}_{mode}" / f"stage{stage}_seed{seed}.jsonl"


def read_rows(task, ds, mode):
    return read_jsonl(sorted((RES / f"{task}_{ds}_{mode}").glob("*.jsonl")))


def key(r):
    return (r["stage"], r["method"], r["seed"], r["p1"], r["p2"])


def score(task, r):
    """validation score used for every selection."""
    return r["val"] if task == "nc" else r["fixed_val_auc"]


def select(task, rows, method, seeds):
    """(p1, p2) with the best mean stage-1 validation score; None until every combo is done on every seed."""
    rs = [r for r in rows if r["stage"] == 1 and r["method"] == method]
    combos = sorted({(r["p1"], r["p2"]) for r in rs})
    grid = GRIDS[method]
    table = {}
    for c in combos:
        v = {r["seed"]: score(task, r) for r in rs if (r["p1"], r["p2"]) == c}
        if all(s in v for s in seeds):
            table[c] = float(np.mean([v[s] for s in seeds]))
    if len(table) < len(grid):
        return None, table
    return max(table, key=table.get), table


GRIDS = {"hetgfd": list(itertools.product(ALPHAS, BETAS)), "pcfi": PCFI_GRID, "knn": KNN_GRID, "svd": SVD_GRID}
# edge-type ranking variants of HetGFD (same alpha x beta grid): label-free probe utility, and every fixed order (oracle)
RANK_VARIANTS = ["hetgfd_umse", "hetgfd_ucos"] + [f"hetgfd_o{i}" for i in range(1, 7)]
GRIDS.update({m: GRIDS["hetgfd"] for m in RANK_VARIANTS})
# synthetic feature injection (Propagate and Inject / FISF): fisf = original (all edge types equal), grid alpha x gamma,
# beta_s = 0.9 (FISF default); hetgfd_inj_* = HetGFD (homophily rank, alpha / beta below) + injection, grid gamma x beta_s
GAMMAS = (0.5, 0.7, 0.9, 1.0)
GRIDS["fisf"] = [(a, gm) for a in ALPHAS for gm in GAMMAS]
GRIDS["hetgfd_inj_attr"] = GRIDS["hetgfd_inj_virt"] = [(gm, bs) for gm in GAMMAS for bs in (0.99, 0.9, 0.5, 0.1)]
# HetGFD alpha / beta used under the injection = the validation choice of server run 09_1 (nc, paper-text HGT) and
# 08 (lp, imputation on the training graph)
INJ_AB = {("nc", "ACM", "entry"): (0.5, 0.2), ("nc", "ACM", "node"): (0.3, 0.1), ("nc", "DBLP", "entry"): (0.3, 0.5),
          ("nc", "DBLP", "node"): (0.3, 0.5), ("nc", "IMDB", "entry"): (0.7, 0.05), ("nc", "IMDB", "node"): (0.7, 0.1),
          ("lp", "ACM", "entry"): (0.3, 0.05), ("lp", "ACM", "node"): (0.3, 0.2), ("lp", "DBLP", "entry"): (0.1, 0.4),
          ("lp", "DBLP", "node"): (0.5, 0.2), ("lp", "IMDB", "entry"): (0.9, 0.4), ("lp", "IMDB", "node"): (0.5, 0.2)}
SEL_METHODS = [m for m in ("hetgfd", "pcfi", "knn", "svd", *RANK_VARIANTS, "fisf", "hetgfd_inj_attr", "hetgfd_inj_virt")
               if m in FINAL_METHODS]  # methods with a stage-1 grid


def n_orders(ds):
    return len(list(itertools.permutations(graph(ds).edges)))


def valid(method, ds):
    """hetgfd_o<i> exists only for i <= number of edge-type orders of the dataset (IMDB has 2 edge types -> 2 orders)."""
    return not method.startswith("hetgfd_o") or int(method[len("hetgfd_o"):]) <= n_orders(ds)


def sel_methods(ds):
    return [m for m in SEL_METHODS if valid(m, ds)]


def final_methods(ds):
    return [m for m in FINAL_METHODS if valid(m, ds)]


def refine_candidates(task, rows, method):
    """top-K stage-1 combinations (by mean validation score); None until stage 1 is complete."""
    sel, table = select(task, rows, method, args.sel_seeds)
    if sel is None:
        return None
    return sorted(table, key=lambda c: -table[c])[: args.refine_top]


def final_choice(task, rows, method):
    """the (p1, p2) used in stage 2: stage-1 winner, or (with --refine-top) the best stage-3 re-evaluation."""
    if not args.refine_top:
        return select(task, rows, method, args.sel_seeds)[0]
    cands = refine_candidates(task, rows, method)
    if cands is None:
        return None
    t3 = {}
    for r in rows:
        if r["stage"] == 3 and r["method"] == method and (r["p1"], r["p2"]) in cands:
            t3.setdefault((r["p1"], r["p2"]), {})[r["seed"]] = score(task, r)
    if any(len(t3.get(c, {})) < len(args.sel_seeds) for c in cands):
        return None
    return max(cands, key=lambda c: np.mean(list(t3[c].values())))


# ---------------------------------------------------------------- one evaluation
G_CACHE, SETUP_CACHE = {}, {}


def graph(ds):
    if ds not in G_CACHE:
        G_CACHE[ds] = load_hgnnac(ds, root=str(W / "data" / "preprocessed"), same_type_directed=True)
    return G_CACHE[ds]


def setup(task, ds, mode, seed):
    k = (task, ds, mode, seed)
    if k not in SETUP_CACHE:
        SETUP_CACHE.clear()  # one (setting, seed) at a time keeps memory flat
        g = graph(ds)
        obs = make_observed_mask_orig(*g.x.shape, RATE, mode, seed=seed, device=dev)
        split = make_orig_split(g, ds, seed) if task == "lp" else None
        SETUP_CACHE[k] = {"obs": obs, "split": split, "rank": None}
    return SETUP_CACHE[k]


def impute(task, ds, mode, seed, method, p1, p2):
    st = setup(task, ds, mode, seed)
    g = st["split"].g_train if (task == "lp" and args.impute_train) else graph(ds)
    obs, x = st["obs"], graph(ds).x.to(dev)
    if method == "zero":
        return torch.where(obs, x, torch.zeros_like(x)), ""
    if method == "fp":
        return fp_orig(g, obs, K, dev), ""
    if method == "pcfi":
        return pcfi_orig(g, obs, p1, p2, K, dev), ""
    if method == "mean":  # missing entries <- mean of the observed values of that channel
        fill = (torch.where(obs, x, torch.zeros_like(x)).sum(0) / obs.float().sum(0).clamp_min(1.0)).expand_as(x)
        return torch.where(obs, x, fill), ""
    if method == "knn":
        return knn_impute(x, obs, int(p1)), ""
    if method == "svd":
        f = x.shape[1]
        return iterative_svd_impute(x, obs, f // 5 if p1 < 1 else f - 1), ""
    if method == "fisf":
        return hetgfd_inject(g, obs, {e: 1 for e in g.edges}, p1, 1.0, p2, 0.9, "attr", K, dev, seed=seed), "all=1"
    if st["rank"] is None:  # edge-type ranking does not depend on (alpha, beta)
        st["rank"] = rank_edge_types(edge_type_homophily(preliminary_diffusion(g, obs, K, dev), g, seed=seed))
    rank = st["rank"]
    if method.startswith("hetgfd_inj_"):
        a, b = INJ_AB[(task, ds, mode)]
        xh = hetgfd_inject(g, obs, rank, a, b, p1, p2, method[len("hetgfd_inj_"):], K, dev, seed=seed)
        return xh, ">".join(sorted(rank, key=rank.get))
    if method in ("hetgfd_umse", "hetgfd_ucos"):
        if "util" not in st:
            st["util"] = probe_utility(g, obs, mode, seed)
        rank = st["util"][method[-4:]]
    elif method.startswith("hetgfd_o"):
        order = list(itertools.permutations(g.edges))[int(method[len("hetgfd_o"):]) - 1]
        rank = {e: k + 1 for k, e in enumerate(order)}
    xh, _ = hetgfd(g, obs, p1, p2, K, dev, seed=seed, rank=rank)
    return xh, ">".join(sorted(rank, key=rank.get))


def probe_utility(g, obs, mode, seed, probe_frac=0.2, n_rounds=3):
    """label-free edge-type utility (local exp04): hide probe_frac of the observed entries (rows for structural missing),
    diffuse (FP+VF: all edge types, unweighted) with every edge type and with each one left out;
    U_r = probe error without r - probe error with all (mse) / probe cosine with all - without r (cos).
    Returns the two rankings (1 = most useful)."""
    x = g.x.to(dev)
    ets = list(g.edges)
    mse, cos = np.zeros((n_rounds, len(ets))), np.zeros((n_rounds, len(ets)))
    for t in range(n_rounds):
        tr, pr = split_probe(obs, probe_frac, mode, seed=seed * 100 + t)
        rows = pr.any(1)

        def run(sub):
            gs = dataclasses.replace(g, edges={e: g.edges[e] for e in sub}, edge_ends={e: g.edge_ends[e] for e in sub})
            xh = relation_aware_diffusion(gs, tr, {e: 1 for e in sub}, 1.0, 1.0, K, dev)
            err = float(((xh - x)[pr] ** 2).mean())
            c = float(torch.nn.functional.cosine_similarity(xh[rows], x[rows], dim=1).mean())
            return err, c

        m_all, c_all = run(ets)
        for i, e in enumerate(ets):
            m, c = run([o for o in ets if o != e])
            mse[t, i], cos[t, i] = m - m_all, c_all - c
    out = {}
    for name, u in (("umse", mse.mean(0)), ("ucos", cos.mean(0))):
        order = [ets[i] for i in np.argsort(-u, kind="stable")]
        out[name] = {e: k + 1 for k, e in enumerate(order)}
    return out


def evaluate(task, ds, mode, seed, xh, layers, lrs):
    if task == "nc":
        na = NONATTR_TEXT[ds] if args.hgt == "text" else HGT_KW["nonattr_feat"]
        kw = {k: v for k, v in HGT_KW.items() if k != "nonattr_feat"}
        r = evaluate_features(graph(ds), xh.cpu(), layers, lrs, seed=seed, device=dev, nonattr_feat=na, **kw, **EPOCH_KW)
        return {"val": r["val_macro"], "test_macro": r["test_macro"], "test_micro": r["test_micro"],
                "layers": r["layers"], "lr": r["lr"]}
    r = evaluate_features_orig(setup(task, ds, mode, seed)["split"], xh.cpu(), layers, lrs, seed=seed, device=dev, **EPOCH_KW)
    return {f"{m}_{k}": r[m][k] for m in ("fixed", "orig") for k in ("val_auc", "test_auc", "test_ap", "layers", "lr")}


def run_one(task, ds, mode, stage, method, seed, p1, p2):
    t0 = time.time()
    Q.status["current"] = f"{task} {ds} {mode} stage{stage} {method} seed={seed} p=({p1},{p2})"
    for attempt in range(3):
        try:
            xh, rank = impute(task, ds, mode, seed, method, p1, p2)
            res = (evaluate(task, ds, mode, seed, xh, SEL_LAYERS, SEL_LRS) if stage == 1
                   else evaluate(task, ds, mode, seed, xh, LAYERS, LRS))
            break
        except torch.cuda.OutOfMemoryError:  # noqa: PERF203
            log.warning(f"GPU out of memory ({Q.status['current']}), attempt {attempt + 1}/3 — waiting 120 s")
            xh = None
            torch.cuda.empty_cache()
            time.sleep(120)
    else:
        log.error(f"skipped after 3 OOM: {Q.status['current']} (will be retried on the next launch)")
        return None
    del xh
    torch.cuda.empty_cache()
    row = dict(task=task, dataset=ds, mode=mode, stage=stage, method=method, seed=seed, p1=p1, p2=p2, rank=rank,
               **res, sec=round(time.time() - t0, 1), host=os.uname().nodename, finished=time.strftime("%Y-%m-%d %H:%M:%S"))
    append_jsonl(unit_path(task, ds, mode, stage, seed), row)
    return row


def fmt(task, r):
    if task == "nc":
        return f"val={r['val']*100:.2f} test macro={r['test_macro']*100:.2f} micro={r['test_micro']*100:.2f}"
    return (f"fixed val={r['fixed_val_auc']*100:.2f} test auc={r['fixed_test_auc']*100:.2f} ap={r['fixed_test_ap']*100:.2f}"
            f" | orig auc={r['orig_test_auc']*100:.2f}")


# ---------------------------------------------------------------- summaries
def summarize(task, ds, mode):
    rows = read_rows(task, ds, mode)
    out = {"task": task, "dataset": ds, "missing": {"node": "structural", "entry": "uniform"}[mode], "n_rows": len(rows),
           "paper_ab": PAPER_AB[task][(ds, mode)], "paper": PAPER[task][(ds, mode)], "stage1": {}, "stage2": {}}
    m1, m2 = ("val", "test_macro") if task == "nc" else ("fixed_val_auc", "fixed_test_auc")
    for method in sel_methods(ds):
        sel, table = select(task, rows, method, args.sel_seeds)
        rs = [r for r in rows if r["stage"] == 1 and r["method"] == method]
        if not rs:
            continue
        tests = {}
        for r in rs:
            tests.setdefault((r["p1"], r["p2"]), []).append(r[m2])
        info = {"n_done": len(rs), "selected": sel,
                "top5_by_val": [[c[0], c[1], round(v * 100, 2), round(float(np.mean(tests[c])) * 100, 2)]
                                for c, v in sorted(table.items(), key=lambda t: -t[1])[:5]],
                "best_test": max(([c[0], c[1], round(float(np.mean(v)) * 100, 2)] for c, v in tests.items()), key=lambda t: t[2])}
        if method == "hetgfd":
            pa = tuple(PAPER_AB[task][(ds, mode)])
            if pa in tests:
                info["paper_ab_test"] = round(float(np.mean(tests[pa])) * 100, 2)
        if args.refine_top:
            cands = refine_candidates(task, rows, method) or []
            r3 = {}
            for r in rows:
                if r["stage"] == 3 and r["method"] == method:
                    r3.setdefault((r["p1"], r["p2"]), []).append(r)
            info["refined"] = {"candidates": cands, "chosen": final_choice(task, rows, method),
                               "full_grid_on_sel_seeds": [[c[0], c[1], len(r3.get(c, [])),
                                                           round(float(np.mean([score(task, x) for x in r3[c]])) * 100, 2) if c in r3 else None,
                                                           round(float(np.mean([x[m2] for x in r3[c]])) * 100, 2) if c in r3 else None]
                                                          for c in cands]}
        out["stage1"][method] = info
    s2 = [r for r in rows if r["stage"] == 2]
    metrics = ("test_macro", "test_micro") if task == "nc" else ("fixed_test_auc", "fixed_test_ap", "orig_test_auc", "orig_test_ap")
    for method in final_methods(ds):
        rs = sorted([r for r in s2 if r["method"] == method], key=lambda r: r["seed"])
        fc = final_choice(task, rows, method) if method in SEL_METHODS else None
        if fc is not None:  # only runs with the final choice (older runs with another choice may sit in the same files)
            rs = [r for r in rs if (r["p1"], r["p2"]) == tuple(fc)]
        if rs:
            out["stage2"][method] = {"n": len(rs), "p": [rs[0]["p1"], rs[0]["p2"]],
                                     **{m: [round(float(np.mean([r[m] for r in rs])) * 100, 2),
                                            round(float(np.std([r[m] for r in rs])) * 100, 2)] for m in metrics},
                                     "per_seed": {r["seed"]: round(r[metrics[0]] * 100, 2) for r in rs}}
    h, p = out["stage2"].get("hetgfd"), out["stage2"].get("pcfi")
    if h and p:
        common = sorted(set(h["per_seed"]) & set(p["per_seed"]))
        out["hetgfd_minus_pcfi"] = {"n": len(common),
                                    "mean_diff": round(float(np.mean([h["per_seed"][s] - p["per_seed"][s] for s in common])), 2) if common else None,
                                    "wins": sum(h["per_seed"][s] > p["per_seed"][s] for s in common)}
    atomic_write(RES / f"summary_{task}_{ds}_{mode}.json", json.dumps(out, indent=1, ensure_ascii=False))
    # CSV: one line per method (final stage), easy to paste back
    hdr = ["task", "hgt", "dataset", "missing", "method", "n_seeds", "p1", "p2"]
    hdr += [f"{m}_{s}" for m in metrics for s in ("mean", "std")] + ["paper_" + metrics[0], "paper_" + metrics[1]]
    lines = [",".join(hdr)]
    for method, s in out["stage2"].items():
        pap = PAPER[task][(ds, mode)].get(method, ("", ""))
        lines.append(",".join(map(str, [task, args.hgt, ds, out["missing"], method, s["n"], s["p"][0], s["p"][1],
                                        *[x for m in metrics for x in s[m]], pap[0], pap[1]])))
    atomic_write(RES / f"summary_{task}_{ds}_{mode}.csv", "\n".join(lines) + "\n")
    return out


# ---------------------------------------------------------------- work units + main loop
SETTINGS = [(t_, d, m) for t_ in args.tasks for d in args.datasets for m in args.modes]


def unit_jobs(u):
    task, ds, mode, stage, seed = u
    if stage == 1:
        return [(m, a, b) for m in sel_methods(ds) for a, b in GRIDS[m]]
    rows = read_rows(task, ds, mode)
    if stage == 3:
        return [(m, a, b) for m in sel_methods(ds) for a, b in (refine_candidates(task, rows, m) or [])]
    sel = {m: final_choice(task, rows, m) for m in sel_methods(ds)}
    return [(m, *(sel[m] if m in sel else (0.0, 0.0))) for m in final_methods(ds)]


def uid(u):
    return "{}_{}_{}_stage{}_seed{}".format(*u)


def build_units():
    fs = args.final_seeds
    units = [(t_, d, m, 1, s) for (t_, d, m) in SETTINGS for s in args.sel_seeds]  # setting-major: GPUs share a setting
    if args.refine_top:
        units += [(t_, d, m, 3, s) for (t_, d, m) in SETTINGS for s in args.sel_seeds]
    for chunk in (fs[:5], fs[5:]):
        units += [(t_, d, m, 2, s) for s in chunk for (t_, d, m) in SETTINGS]
    return units


def main():
    global Q
    Q = Queue(STATE, args.launch, args.worker)
    log.info(f"==== worker {args.worker} (launch {args.launch}) start | CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} | "
             f"torch {torch.__version__} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | "
             f"sel seeds {args.sel_seeds} | final seeds {args.final_seeds} | methods {FINAL_METHODS} | hgt {args.hgt} | refine top {args.refine_top} | smoke={args.smoke}")
    units = build_units()
    by_id = {uid(u): u for u in units}

    def done_keys(u):
        task, ds, mode, stage, seed = u
        return {(r["method"], r["p1"], r["p2"]) for r in read_jsonl([unit_path(task, ds, mode, stage, seed)])}

    def is_ready(i):
        u = by_id[i]
        if u[3] == 1:
            return True
        rows = read_rows(*u[:3])
        if u[3] == 3:
            return all(refine_candidates(u[0], rows, m) is not None for m in sel_methods(u[1]))
        return all(final_choice(u[0], rows, m) is not None for m in sel_methods(u[1]))

    def is_done(i):
        u = by_id[i]
        if not is_ready(i):
            return False
        return all(tuple(j) in done_keys(u) for j in unit_jobs(u))

    def deps(i):
        u = by_id[i]
        if u[3] == 1:
            return []
        d = [uid((*u[:3], 1, s)) for s in args.sel_seeds]
        return d + ([uid((*u[:3], 3, s)) for s in args.sel_seeds] if u[3] == 2 and args.refine_top else [])

    def run_unit(i):
        u = by_id[i]
        task, ds, mode, stage, seed = u
        done = done_keys(u)
        todo = [j for j in unit_jobs(u) if tuple(j) not in done]
        log.info(f"---- unit {i}: {len(todo)} runs to do")
        for n, (m, p1, p2) in enumerate(todo, 1):
            try:
                r = run_one(task, ds, mode, stage, m, seed, p1, p2)
            except Exception:
                log.error(f"error in {i} {m} ({p1},{p2}):\n{traceback.format_exc()}")
                continue
            if r is not None:
                log.info(f"[{i} {n}/{len(todo)}] {m} ({p1},{p2}) {fmt(task, r)} ({r['sec']:.0f}s)")
                summarize(task, ds, mode)

    run_queue(Q, [uid(u) for u in units], is_done, is_ready, run_unit, deps, log)


if __name__ == "__main__":
    main()


In [ ]:
# ===== 셀 6: GPU 확인 + 작업자(GPU당 1개) 실행 도구 정의 — 탭을 다시 열 때도 셀 1 다음에 실행 =====
import subprocess, signal, glob, socket
# 작업자 이름 = <이 작업(노드) 이름>-gpu<i>: 같은 WORK 폴더를 여러 작업(job)이 함께 써도 이름이 겹치지 않고,
# 먼저 돌던 작업의 launch id를 이어받아 같은 대기열에서 일을 나눠 가짐 (같은 일을 두 번 하지 않음)
HOST = os.environ.get("JOB_TAG") or socket.gethostname().split(".")[0]
import torch
print(subprocess.run("nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv", shell=True,
                     capture_output=True, text=True).stdout)
_BASE = [x for x in os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",") if x.strip()]  # 작업 스케줄러가 정해 둔 GPU 목록

def gpu_list():
    n = torch.cuda.device_count()
    return list(range(n)) if GPUS == "auto" else list(GPUS)

def physical(i):  # i번째로 보이는 GPU → 작업자에게 줄 CUDA_VISIBLE_DEVICES 값
    return _BASE[i] if _BASE else str(i)

def _gpu_uuids():
    try:
        out = subprocess.run("nvidia-smi --query-gpu=index,uuid --format=csv,noheader", shell=True,
                             capture_output=True, text=True).stdout
        return [tuple(x.strip() for x in l.split(",")) for l in out.splitlines() if "," in l]
    except Exception:
        return []
_UUIDS = _gpu_uuids()

def wname(i):
    """작업자 이름 = GPU 고유번호(UUID) 끝 8자리 → 작업(job)·노드 이름이 같아도 GPU가 다르면 절대 겹치지 않고,
    같은 GPU에서 다시 시작하면 같은 이름 (이어 돌리기)."""
    p = physical(i)
    u = p if p.startswith("GPU-") else dict(_UUIDS).get(p) or (_UUIDS[i][1] if i < len(_UUIDS) else "")
    tag = "".join(ch for ch in u if ch.isalnum())[-8:] if u else f"{HOST}{p}"
    return f"gpu-{tag}"

def worker_state(name, out=""):
    try: return json.loads(pathlib.Path(WORK, out, "state", f"{name}.json").read_text())
    except Exception: return None

def _pid_alive(pid):
    try: os.kill(int(pid), 0); return True
    except (OSError, ValueError, TypeError): return False

def alive_workers(out=""):
    """최근 2분 안에 신호를 보낸 작업자. 이 작업의 GPU에 붙은 작업자는 프로세스가 실제로 살아 있는지도 확인
    (멈추기 직후에는 신호 파일이 아직 '최근'이라 살아 있는 것으로 착각할 수 있음)."""
    res = {}
    for f in pathlib.Path(WORK, out, "state").glob("*gpu*.json"):
        s = worker_state(f.stem, out)
        if not s or s.get("current") in ("finished", "stopped") or time.time() - s["time"] >= 120:
            continue
        if f.stem in {wname(i) for i in gpu_list()} and not _pid_alive(s.get("pid")):
            continue   # 이 작업의 GPU에 붙은 작업자인데 프로세스가 없음 = 죽은 것 (멈추기 직후 등)
        res[f.stem] = s
    return res

def launch(script_args, out=""):
    """GPU마다 작업자 1개. 이미 살아 있는 작업자는 그대로 두고, 없는 GPU에만 새로 띄움 (같은 launch id로 일을 나눔)."""
    missing = [s for s in RUN.split()[1:2] if not pathlib.Path(s).exists()]
    if missing or not pathlib.Path(WORK, "src", "gpu_queue.py").exists():
        print("소스 파일이 없습니다:", missing or [str(pathlib.Path(WORK, "src", "gpu_queue.py"))])
        print("→ 셀 5의 %%writefile 셀들을 모두 실행한 뒤 이 셀을 다시 실행하세요 (작업자를 띄우지 않았습니다).")
        return
    for d in ("logs", "state", "results"):
        pathlib.Path(WORK, out, d).mkdir(parents=True, exist_ok=True)
    alive = alive_workers(out)
    started = {}
    lid = next(iter(alive.values()))["launch"] if alive else time.strftime("L%Y%m%d_%H%M%S")
    for i in gpu_list():
        name = wname(i)
        if name in alive:
            print(f"{name}: 이미 실행 중 → 그대로 둠 | {alive[name].get('current')}"); continue
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": physical(i), "PYTHONWARNINGS": "ignore", "OMP_NUM_THREADS": "4",
               "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
        cmd = (f"{RUN} --workdir {WORK} --worker {name} --launch {lid} {script_args} "
               f">> {pathlib.Path(WORK, out, 'logs', name + '.out')} 2>&1")
        p = subprocess.Popen(["bash", "-c", cmd], env=env, start_new_session=True,
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        pathlib.Path(WORK, out, "state", f"{name}.pgid").write_text(str(p.pid))
        started[name] = p
        print(f"{name}: 시작 (CUDA_VISIBLE_DEVICES={physical(i)}, pid {p.pid}, launch {lid})")
    if started:  # 바로 죽는 작업자가 없는지 최대 3분 확인
        print("작업자가 정상적으로 시작되는지 확인 중 (최대 3분)...")
        t0 = time.time()
        while time.time() - t0 < 180:
            time.sleep(10)
            dead = {n for n, p in started.items() if p.poll() is not None}
            states = {n: (worker_state(n, out) or {}).get("current", "") for n in started}
            if dead or all(s and not s.startswith("starting") for s in states.values()):
                break
        for n, p in started.items():
            s = (worker_state(n, out) or {}).get("current", "-")
            if p.poll() is not None and s != "finished":
                print(f"[{n}] 시작 직후 종료됨 (exit {p.returncode}) — 아래 오류를 Claude에게 붙여넣어 주세요")
                print("\n".join(pathlib.Path(WORK, out, "logs", n + ".out").read_text().strip().splitlines()[-25:]))
            else:
                print(f"[{n}] 정상 | {s}")

def stop_previous(script, workdirs, out=""):
    """이 WORK(와 workdirs)로 돌던 이전 프로세스를 모두 멈춤. 지금 살아 있는 작업자(같은 launch)만 남김.
    대상: 예전 버전 노트북의 프로세스, 신호가 끊긴(멈춘) 작업자, 다른 launch에서 남은 작업자."""
    alive = alive_workers(out)
    keep = [(f"--worker {n} ", f"--launch {s['launch']}") for n, s in alive.items()]
    for _ in range(6):   # 작업 줄(bash)이 새 프로세스를 띄울 수 있으므로 없어질 때까지 반복
        lines = subprocess.run("ps -u $(whoami) -o pid=,args=", shell=True, capture_output=True, text=True).stdout.splitlines()
        targets = [l for l in lines
                   if any(f"{script} --workdir {w} " in l + " " for w in workdirs)
                   and not any(a in l + " " and b in l for a, b in keep) and int(l.split()[0]) != os.getpid()]
        if not targets:
            return
        for l in targets:
            print("이전 프로세스 멈춤:", l.strip()[:160])
            try: os.kill(int(l.split()[0]), signal.SIGTERM)
            except ProcessLookupError: pass
        time.sleep(5)
    print("경고: 이전 프로세스가 계속 남아 있음 — 이 셀 출력을 Claude에게 붙여넣어 주세요")

def stop(out=""):
    for i in gpu_list():   # 이 작업이 띄운 작업자만 (다른 작업의 작업자는 건드리지 않음)
        f = pathlib.Path(WORK, out, "state", f"{wname(i)}.pgid")
        if not f.exists(): continue
        try: os.killpg(int(f.read_text()), signal.SIGTERM); print("멈춤:", f.stem)
        except (ProcessLookupError, ValueError): print("이미 끝남:", f.stem)
        s = worker_state(f.stem, out)
        if s and s.get("current") != "finished":   # 바로 다시 시작해도 '실행 중'으로 착각하지 않게 표시
            s["current"] = "stopped"; s["time"] = time.time()
            pathlib.Path(WORK, out, "state", f"{f.stem}.json").write_text(json.dumps(s))

def show_workers(out=""):
    print(time.strftime("%H:%M:%S"))
    print(subprocess.run("nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip())
    files = sorted(pathlib.Path(WORK, out, "state").glob("*gpu*.json"))
    if not files:
        print("작업자 기록 없음 — 시작 셀을 아직 실행하지 않았거나, 작업자가 시작 직후 죽었습니다.")
        for o in sorted(pathlib.Path(WORK, out, "logs").glob("*gpu*.out")):
            print(f"## {o.name} 마지막 15줄"); print("\n".join(o.read_text().strip().splitlines()[-15:]))
    for f in files:
        s = worker_state(f.stem, out)
        if not s: continue
        age = time.time() - s["time"]
        st = ("끝남" if s.get("current") == "finished" else "멈춤(멈추기 셀)" if s.get("current") == "stopped"
              else "실행 중" if f.stem in alive_workers(out) else f"멈춤({age/60:.0f}분 전 마지막 신호)")
        print(f"[{f.stem}] {st} | {s.get('current')}")

print("노트북 버전: 2026-10-11 14:11 (작업자 이름이 gpu-xxxxxxxx 형식이면 최신; gpu0 형식이면 이전 파일)")
print("보이는 GPU:", gpu_list(), "→ 작업자에게 줄 번호:", [physical(i) for i in gpu_list()], "→ 작업자 이름:", [wname(i) for i in gpu_list()])
if not gpu_list():
    print("경고: 이 커널에서 GPU가 보이지 않습니다 — 이 출력을 Claude에게 붙여넣어 주세요")

In [ ]:
# ===== 셀 7 (처음 한 번만, 선택): 짧은 점검 — IMDB, 아주 작은 격자·에폭, 첫 번째 GPU로 이 셀에서 직접 실행 (수 분) =====
# 결과는 WORK/smoke 에 따로 저장. 마지막에 "finished (no more work)"가 나오면 정상
env = {**os.environ, "CUDA_VISIBLE_DEVICES": physical(gpu_list()[0]), "PYTHONWARNINGS": "ignore"}
r = subprocess.run(f"{RUN} --workdir {WORK} --worker gpu0 --launch smoke_{time.time():.0f} --smoke --tasks lp --datasets IMDB --modes entry --methods fisf --impute-train --refine-top 5 --datasets IMDB --modes node "
                   f"--sel-seeds 0 --final-seeds 0", shell=True, env=env, capture_output=True, text=True)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "sparse_csr" not in l)[-4000:])

In [ ]:
# ===== 셀 8: 시작 / 이어서 시작 — GPU마다 작업자 1개 (새 작업을 잡을 때마다 실행; 이미 도는 작업자는 그대로 둠) =====
# 이전에 돌던 프로세스(예전 02 노트북의 ~/hetgfd_hgt_final 실행 포함)를 먼저 모두 멈춤 — 끝난 학습 기록은 그대로 이어서 씀
stop_previous("run_hgt_final.py", [WORK, os.path.expanduser("~/hetgfd_hgt_final")])
launch(ARGS)
# 이 셀이 끝나면(최대 3분) 셀 9로 확인

In [ ]:
# ===== 셀 9: 진행 상황 (탭을 다시 열면: 셀 1 → 셀 6 → 이 셀) =====
def read_jsonl(paths):
    out = []
    for p in paths:
        for l in (p.read_text().splitlines() if p.exists() else []):
            try: out.append(json.loads(l))
            except Exception: pass
    return out
show_workers()
n1, n2 = len(SEL_SEEDS) * 20, 1 * len(FINAL_SEEDS)
print(f"\n{'설정':22s} 1단계(/{n1})  재평가  2단계(/{n2})  마지막 기록")
for t in ["lp"]:
    for ds in ["IMDB"]:
        for m in ["node", "entry"]:
            rows = read_jsonl(sorted(pathlib.Path(WORK, "results", f"{t}_{ds}_{m}").glob("*.jsonl")))
            a = sum(r["stage"] == 1 for r in rows); b = sum(r["stage"] == 2 for r in rows); c3 = sum(r["stage"] == 3 for r in rows)
            last = max((r["finished"] for r in rows), default="-")
            print(f"{t+' '+ds+' '+m:22s} {a:5d}     {c3:5d}    {b:4d}       {last}")
for f in sorted(pathlib.Path(WORK, "logs").glob("*gpu*.log")):
    print(f"\n--- {f.name} 마지막 2줄"); print("\n".join(f.read_text().strip().splitlines()[-2:]))
# 오류 확인: !grep -n "error\|Traceback" {WORK}/logs/*gpu*.log {WORK}/logs/*gpu*.out | tail

In [ ]:
# ===== 셀 9-1 (선택): 자동 새로고침 (60초마다). 멈추려면 ■ — 실험은 계속 돈다 =====
from IPython.display import clear_output
_CELL = In[[i for i, s in enumerate(In) if s.startswith("# ===== 셀 9:")][-1]]
try:
    while True:
        clear_output(wait=True); exec(_CELL); time.sleep(60)
except KeyboardInterrupt:
    print("자동 새로고침 멈춤 (실험은 계속 실행 중)")

In [ ]:
# ===== (필요 시) 멈추기 — 셀 8을 다시 실행하면 멈춘 곳부터 이어짐 =====
stop()

In [ ]:
# ===== 셀 10: 결과 요약 — 이 셀의 출력 전체를 복사해서 Claude에게 붙여넣기 =====
out = {}
for f in sorted(pathlib.Path(WORK, "results").glob("summary_*.json")):
    s = json.loads(f.read_text())
    out[f"{s['task']}_{s['dataset']}_{s['missing']}"] = {k: s[k] for k in ("n_rows", "paper_ab", "stage1", "stage2", "hetgfd_minus_pcfi") if k in s}
    for m in out[f"{s['task']}_{s['dataset']}_{s['missing']}"].get("stage2", {}).values():
        m.pop("per_seed", None)
print(json.dumps(out, ensure_ascii=False, separators=(",", ":")))

In [ ]:
# ===== 셀 11 (Claude가 요청할 때): 원자료 전체를 한 파일로 묶기 → 파일 탐색기에서 내려받아 전달 =====
import tarfile
arc = pathlib.Path(WORK, f"hgt_final_results_{time.strftime('%Y%m%d_%H%M')}.tar.gz")
with tarfile.open(arc, "w:gz") as t:
    t.add(pathlib.Path(WORK, "results"), arcname="results"); t.add(pathlib.Path(WORK, "logs"), arcname="logs")
print(arc, f"{arc.stat().st_size/1e6:.1f} MB")

In [ ]:
# ===== 셀 12: CSV 만들기 — 요약 CSV 출력 전체를 복사해서 Claude에게 붙여넣기 (파일은 WORK/export/) =====
# summary.csv : 설정 × 방법별 최종(2단계) 평균·표준편차 + 논문 수치
# selection.csv: 1단계에서 고른 하이퍼파라미터와 검증 상위 5개
# runs.csv     : 모든 학습 1회 결과 (크면 아래 마지막 줄 주석을 풀어 조각으로 출력)
import csv, io
R = pathlib.Path(WORK, "results"); X = pathlib.Path(WORK, "export"); X.mkdir(exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M")
summ, hdr = [], None   # 노드 분류·링크 예측은 열이 달라서 머리줄이 바뀔 때마다 새로 씀
for f in sorted(R.glob("summary_*.csv")):
    ls = f.read_text().splitlines()
    if not ls: continue
    if ls[0] != hdr: summ.append(ls[0]); hdr = ls[0]
    summ += ls[1:]
(X / f"summary_{stamp}.csv").write_text("\n".join(summ) + "\n")
sel = ["setting,method,selected_p1,selected_p2,n_complete,rank,p1,p2,val_mean,test_mean"]
for f in sorted(R.glob("summary_*.json")):
    s = json.loads(f.read_text()); name = f.stem[len("summary_"):]
    for m, v in s.get("stage1", {}).items():
        for i, (a, b, vv, tt) in enumerate(v.get("top5_by_val", []), 1):
            sp = v.get("selected") or ["", ""]
            sel.append(",".join(map(str, [name, m, sp[0], sp[1], v.get("n_done", ""), i, a, b, vv, tt])))
(X / f"selection_{stamp}.csv").write_text("\n".join(sel) + "\n")
rows = []
for d in sorted(p for p in R.iterdir() if p.is_dir()):
    for f in sorted(d.glob("*.jsonl")):
        for l in f.read_text().splitlines():
            try: rows.append(json.loads(l))
            except Exception: pass
cols = [c for c in ["task", "dataset", "mode", "stage", "method", "seed", "p1", "p2", "val", "test_macro", "test_micro",
                    "fixed_val_auc", "fixed_test_auc", "fixed_test_ap", "orig_val_auc", "orig_test_auc", "orig_test_ap",
                    "layers", "lr", "fixed_layers", "fixed_lr", "sec"] if any(c in r for r in rows)]
buf = io.StringIO(); w = csv.writer(buf); w.writerow(cols)
for r in rows:
    w.writerow([round(r[c], 4) if isinstance(r.get(c), float) else r.get(c, "") for c in cols])
(X / f"runs_{stamp}.csv").write_text(buf.getvalue())
print(f"# 저장: {X}/summary_{stamp}.csv, selection_{stamp}.csv, runs_{stamp}.csv ({len(rows)}행, {len(buf.getvalue())/1000:.0f} KB)")
print("# ===== summary.csv ====="); print("\n".join(summ))
print("# ===== selection.csv ====="); print("\n".join(sel))
# 전체 실행 CSV도 필요하면 (Claude가 요청할 때) 아래 주석을 풀어 3만 글자씩 조각으로 출력:
# txt = buf.getvalue(); [print(f"===== runs 조각 {i//30000+1} =====\n" + txt[i:i+30000]) for i in range(0, len(txt), 30000)]